# Transformer Made Simple (draft)

(Attila M. Magyar, 2026)

## Abstract

 * The scaled dot-product attention mechanism of the Transformer architecture \[[1](#ref-1)\] is mathematically equivalent to an ordinary MLP with a $\text{softmax}$ activation that is evaluated on the products of input matrices.
 * The ReLUFormer architecture \[[5](#ref-5)\] replaces $\text{softmax}$ with $\text{ReLU}$ and achieves similar or even better performance than the vanilla Transformer.
 * Preliminary experiments: WIP

## Table of contents

 * [Abstract](#Abstract)
 * [Table of contents](#Table-of-contents)
 * [1 Introduction](#1-Introduction)
    * [1.1 Notation](#1.1-Notation)
    * [1.2 Fully connected feed-forward neural network](#1.2-Fully-connected-feed-forward-neural-network)
       * [1.2.1 Parallel version](#1.2.1-Parallel-version)
       * [1.2.2 Universal approximation theorem](#1.2.2-Universal-approximation-theorem)
    * [1.3 Multi-head scaled dot-product attention \(QKV-attention\)](#1.3-Multi-head-scaled-dot-product-attention-\(QKV-attention\))
       * [1.3.1 Self-attention \(transformer encoder\)](#1.3.1-Self-attention-\(transformer-encoder\))
       * [1.3.2 Cross-attention \(transformer decoder\)](#1.3.2-Cross-attention-\(transformer-decoder\))
       * [1.3.3 Usage in language models](#1.3.3-Usage-in-language-models)
       * [1.3.4 Masked attention](#1.3.4-Masked-attention)
 * [2 A general interpretation of the scaled dot-product attention](#2-A-general-interpretation-of-the-scaled-dot-product-attention)
    * [2.1 Notes](#2.1-Notes)
 * [3 Simplifications](#3-Simplifications)
    * [3.1 Replacing softmax with ReLU](#3.1-Replacing-softmax-with-ReLU)
    * [3.2 Simplifying the attention computation](#3.2-Simplifying-the-attention-computation)
       * [3.2.1 Eliminating the K and V projections](#3.2.1-Eliminating-the-K-and-V-projections)
       * [3.2.2 Eliminating the constant scaling](#3.2.2-Eliminating-the-constant-scaling)
       * [3.2.3 A computationally cheaper alternative](#3.2.3-A-computationally-cheaper-alternative)
    * [3.3 Drop-in](#3.3-Drop-in)
 * [4 Experiments](#4-Experiments)
    * [4.1 Model variants](#4.1-Model-variants)
    * [4.2 Drop-in schedule](#4.2-Drop-in-schedule)
    * [4.3 Classification tasks](#4.3-Classification-tasks)
       * [4.3.1 AG's News](#4.3.1-AG's-News)
       * [4.3.2 CIFAR-10](#4.3.2-CIFAR-10)
       * [4.3.3 CIFAR-100](#4.3.3-CIFAR-100)
       * [4.3.4 IMDb](#4.3.4-IMDb)
       * [4.3.5 ListOps](#4.3.5-ListOps)
    * [4.4 Text generation](#4.4-Text-generation)
       * [4.4.1 IMDb-gen](#4.4.1-IMDb-gen)
       * [4.4.2 WikiText2](#4.4.2-WikiText2)
 * [5 Results](#5-Results)
    * [5.1 Discussion](#5.1-Discussion)
 * [6 Future work](#6-Future-work)
 * [References](#References)
 * [License](#License)
 * [Appendix: Code](#Appendix:-Code)
    * [Dependencies](#Dependencies)
    * [Default device](#Default-device)
    * [Paths](#Paths)
    * [Log](#Log)
    * [Debug](#Debug)
    * [Random seed](#Random-seed)
    * [Model components](#Model-components)
       * [Attention](#Attention)
       * [Drop-in](#Drop-in)
       * [Transformer](#Transformer)
       * [Vision transformer](#Vision-transformer)
       * [Image processing](#Image-processing)
       * [Text processing](#Text-processing)
    * [Snapshot management](#Snapshot-management)
    * [Training loop](#Training-loop)
    * [Batches](#Batches)
    * [Loss functions](#Loss-functions)
    * [Models](#Models)
       * [Causal language model](#Causal-language-model)
       * [Image classifier](#Image-classifier)
       * [Text classifier](#Text-classifier)
    * [Metrics](#Metrics)
    * [Datasets](#Datasets)
       * [Utilities](#Utilities)
       * [AG's News dataset](#AG's-News-dataset)
       * [CIFAR-10 dataset](#CIFAR-10-dataset)
       * [CIFAR-100 dataset](#CIFAR-100-dataset)
       * [IMDb dataset](#IMDb-dataset)
       * [ListOps dataset](#ListOps-dataset)
       * [WikiText2 dataset](#WikiText2-dataset)
    * [Experiments](#Experiments)
       * [AG's News experiment](#AG's-News-experiment)
       * [CIFAR-10 experiment](#CIFAR-10-experiment)
       * [CIFAR-100 experiment](#CIFAR-100-experiment)
       * [IMDb experiment](#IMDb-experiment)
       * [IMDb-gen experiment](#IMDb-gen-experiment)
       * [ListOps experiment](#ListOps-experiment)
       * [WikiText2 experiment](#WikiText2-experiment)
    * [Results summary](#Results-summary)

## 1 Introduction

The famous *Attention Is All You Need* paper \[[1](#ref-1)\] introduced the Transformer architecture, a neural network based on the multi-head scaled dot-product attention mechanism (also known as query-key-value attention), for natural language translation. These models later became known to be highly effective in several other machine learning fields, including other language processing tasks, computer vision \[[2](#ref-2)\], audio processing \[[3](#ref-3)\], etc. Therefore, it is worth sudying their operation from a more general perspective than language processing or sequence modeling.

### 1.1 Notation

We are going to use row-major notation: $xA = b \in \mathbb{R}^m$ for $m, n \in \mathbb{N}$ and $A \in \mathbb{R}^{n \times m}$ and $x \in \mathbb{R}^{1 \times n}$ (or $x \in \mathbb{R}^n$ for short).

### 1.2 Fully connected feed-forward neural network

For a model dimensionality $d_m \in \mathbb{N}$, let $x \in \mathbb{R}^{d_m}$ be an input vector. A fully connected feed-forward neural network with 2 layers computes the following expression:

<a id="eq-1"></a>

$$
y = f^{(1)}\left( x \cdot W^{(1)} + b^{(1)} \right) \cdot W^{(2)}
    + b^{(2)} \quad \in \mathbb{R}^{d^{(2)}}
  \qquad\quad(1)
$$

where, for some $d^{(1)},\ d^{(2)} \in \mathbb{N}$:

 * $W^{(1)} \in \mathbb{R}^{d_m \times d^{(1)}}$ and $W^{(2)} \in \mathbb{R}^{d^{(1)} \times d^{(2)}}$ are weight matrices,

 * $b^{(1)} \in \mathbb{R}^{d^{(1)}}$ and $b^{(2)} \in \mathbb{R}^{d^{(2)}}$ are bias vectors,

 * $f^{(1)} : \mathbb{R}^{d^{(1)}} \rightarrow \mathbb{R}^{d^{(1)}}$ is some non-linear function, e.g. $\text{ReLU}(x) = \max(0, x)$.

#### 1.2.1 Parallel version

For some $d_s \in \mathbb{N}$ let's replace $x$ by $X \in \mathbb{R}^{d_s \times d_m}$ and broadcast $b^{(k)}$ into $d_s$ dimensions in order to evaluate the network for multiple input samples simultaneously (with $d_m$ features and $d_s$ samples):

<a id="eq-2"></a>

$$
Y = f^{(1)}\left( X \cdot W^{(1)} + B^{(1)} \right) \cdot W^{(2)}
      + B^{(2)} \quad \in \mathbb{R}^{d_s \times d^{(2)}}
  \qquad\quad(2)
$$

where $B^{(k)} = \left[ b^{(k)} \right]_{i=1}^{d_s} \in \mathbb{R}^{d_s \times d^{(k)}}$.

Note: treating $X$ as a $d_s$ long sequence instead of an unordered collection and using different, position-dependent vectors in $B^{(1)}$ is analogous to positional encoding in Transformers \[[1](#ref-1)\].

#### 1.2.2 Universal approximation theorem

The gist of it is that the power of neural networks comes from the fact that any continuous function can be approximated well by some neural network - and many real-world problems can be mapped to continuous functions, i.e. functions where a small change in the argument leads to a small change in the function's value.

### 1.3 Multi-head scaled dot-product attention (QKV-attention)

The heart and soul of the Transformer \[[1](#ref-1)\] (besides residual connections \[[4](#ref-4)\] and a few other tricks) is the multi-head scaled dot-product attention, with optional masking (depending on the problem).

If $h \in \mathbb{N}$ denotes the number of attention heads in a single attention layer, then the $j$-th head (for $j \in \mathbb{N},\ 1 \le j \le h$) calculates the following expression (row-major notation):

For some $d_m,\ d_n,\ d_q,\ d_{kv} \in \mathbb{N}$ and $X^Q \in \mathbb{R}^{d_q \times d_m}$ and $X^K,\ X^V \in \mathbb{R}^{d_{kv} \times d_n}$ input matrices:

<a id="eq-3"></a>

$$
\begin{align*}
  \hat{Y}_j
    = & \text{Attention}(Q_j,\ K_j,\ V_j)
      & \quad \in \mathbb{R}^{d_q \times d^V}
      & \qquad\quad(3) \\
    = & \text{softmax}\left( \frac{Q_j \cdot K_j^T}{\sqrt{d^K}} + M \right)
        \cdot V_j
      & &
\end{align*}
$$

where:

 * $Q_j = X^Q \cdot W_j^Q$,

 * $K_j = X^K \cdot W_j^K$,

 * $V_j = X^V \cdot W_j^V$,

 * $W_j^Q \in \mathbb{R}^{d_m \times d^K}$, $W_j^K \in \mathbb{R}^{d_n \times d^K}$ and $W_j^V \in \mathbb{R}^{d_n \times d^V}$ are learnable weights for some $d^K,\ d^V \in \mathbb{N}$,

 * usually, $h$, $d^K$, and $d^V$ are chosen so that $h$ is a divisor of both $d_m$ and $d_n$, and then $d^K = \frac{d_m}{h}$ and $d^V = \frac{d_n}{h}$; often $d_m = d_n$ as well,

 * $\text{softmax}$ is taken row-wise independently,

 * and $M \in \mathbb{R}^{d_q \times d_{kv}}$ is a constant called the "_mask_" and is used for specifically suppressing certain attention weights by adding $-\infty$ (or alternatively, a large magnitude negative value) to them, which will then be turned into $0$ by the $\text{softmax}$ function, e.g. during training autoregressive models; it can be omitted when not necessary by setting $M = 0 \in \mathbb{R}^{d_q \times d_{kv}}$.

The resulting $\hat{Y}_j$ matrices from the individual attention heads are then concatenated to form a $d_q \times \left( h \cdot d^V \right)$ dimensional matrix which then gets projected once more by a learnable $W^O \in \mathbb{R}^{\left( h \cdot d^V \right) \times d_m}$ matrix before finally reaching a residual connection where it is added to $X^Q$, and their sum is propagated through the rest of the network:

<a id="eq-4"></a>

$$
Y = \text{Concat}\left( \hat{Y}_1, \dots, \hat{Y}_h \right) \cdot W^O
  \quad \in \mathbb{R}^{d_q \times d_m}
  \qquad\quad(4)
$$

This, of course, is equivalent to splitting $W^O$ into $h$ smaller $W_j^O \in \mathbb{R}^{d^V \times d_m}$ matrices, multiplying each $\hat{Y}_j$ matrix by the corresponding $W_j^O$ matrix, and then summing the results. Let $Y_j = \hat{Y}_j \cdot W_j^O$, then:

<a id="eq-5"></a>

$$
\begin{align*}
Y & = \text{Concat}\left( \hat{Y}_1, \dots, \hat{Y}_h \right) \cdot W^O
  & \quad \in \mathbb{R}^{d_q \times d_m}
  \qquad\quad(5) \\
  & = \sum_{j=1}^h \hat{Y}_j \cdot W_j^O & \\
  & = \sum_{j=1}^h Y_j &
\end{align*}
$$

Note: writing the formula like this makes it easier to rearrange and get a detailed picture of how the attention mechanism works. In practical implementations, $X^Q$, $X^K$, and $X^V$ are projected first, then the projections are split and processed by the individual attention heads, their results are concatenated, and finally, they are projected by a single $W^O$ matrix.

#### 1.3.1 Self-attention (transformer encoder)

For some $d_s \in \mathbb{N}$ and $X \in \mathbb{R}^{d_s \times d_m}$, let $X^Q = X^K = X^V = X$ (with $d_n = d_m$ and $d_q = d_{kv} = d_s$).

#### 1.3.2 Cross-attention (transformer decoder)

For some $d_1,\ d_2 \in \mathbb{N}$ and $X_1 \in \mathbb{R}^{d_1 \times d_m}$ and $X_2 \in \mathbb{R}^{d_2 \times d_n}$ pair of inputs, let $X^Q = X_1$ and let $X^K = X^V = X_2$ (with $d_q = d_1$ and $d_{kv} = d_2$).

#### 1.3.3 Usage in language models

A language model works with a sequence of language fragments called tokens which can be words, n-grams, letters, etc.

The first component of a language model is the token embedding layer which maps the tokens to vectors in a $d_m$-dimensional space called the embedding space or latent space. The arrangement of these vectors within the space usually turns out to reflect their semantical and grammatical relationships.

Once the entire token sequence is converted to a matrix representing a sequence of vectors, a so-called positional encoding matrix is added: this nudges each token vector by a small delta that is unique to each position within the sequence, helping the model identify which token appears at which position within the sequence.

After the embedding and positional encoding, the token sequence begins its journey through the attention blocks of the Transformer. 

The purpose of a single attention head within the attention mechanism is to let the token vectors influence each other's meaning by:

 * calculating their pairwise similarities as their dot products,

 * then turning those similarity scores into a probability distribution via the $\text{softmax}$ function,

 * and finally, using the resulting matrix as sets of weights for calculating each new sequence element as a weighted average of the input token vectors.

Specifically, in a text translation setup, the cross-attention mechanism is said to:

 * extract semantic information from the source language tokens and build a key-value database from it ($K$ and $V$),

 * turn the target language tokens generated so far into a set of queries ($Q$),

 * then compare the queries against the keys by their semantic similarity ($Q \cdot K^T$),

 * and finally, calculate a weighted sum of the values such that the weights are proportional to the similarities of the corresponding keys and queries.

#### 1.3.4 Masked attention

##### 1.3.4.1 Causal mask

Since the output of the transformer is a sequence of vectors (which correspond to tokens in a language model), its training as an autoregressive model can be parallelized: the very last token in the output sequence will be the predicted next token for the whole sequence, and the preceding $d_q - 1$ tokens should match the last $d_q - 1$ tokens of the input. This way the training can update the model's weights for the combined losses of $d_q$ token predictions in a single training step.

For this to work, it must be disallowed for the $n$-th target token to influence the predictions for the preceding $n - 1$ output tokens during training. This is achieved by masking out the unwanted value-weights: by adding an upper-triangle matrix $M$ containing $-\infty$ values (or alternatively, large magnitude negative values) above the diagonal to the argument of the $\text{softmax}$ function, the unwanted weights will become zero.

##### 1.3.4.2 Padding mask

Neural networks are usually trained on multiple samples - called a batch or minibatch - simultaneously. For technical reasons, all the samples in a single batch must have the exact same dimensions, including the sequence lengths. Thus, $d_q$ and $d_{kv}$ for a single batch are determined by the longest sample in that batch, and the rest of the samples are usually padded.

However, assigning attention weights for the padding tokens is undesired, so they are usually also masked using $M$, similarly to causal masking.

## 2 A general interpretation of the scaled dot-product attention

After substitutions, and utilizing the associativity of matrix multiplication and the $(A \cdot B)^T = B^T \cdot A^T$ identity, the $Y_j$ matrices in equation [5](#eq-5) take the following form:

<a id="eq-6"></a>

$$
\begin{align*}
Y_j
  & = \text{Attention}\left(
        X^Q \cdot W_j^Q, \quad X^K \cdot W_j^K, \quad X^V \cdot W_j^V
      \right)
      \cdot W_j^O
      & \quad \in \mathbb{R}^{d_q \times d_m}
      & \qquad\quad(6) \\
  & = \left(
        \text{softmax}\left(
          \frac{
            \left( X^Q \cdot W_j^Q \right) \cdot \left( X^K \cdot W_j^K \right)^T
          }{\sqrt{d^K}} + M
        \right)
        \cdot X^V \cdot W_j^V
      \right)
      \cdot W_j^O
      & & \\
  & = \text{softmax} \left(
          X^Q
          \cdot \left( \frac{1}{\sqrt{d^K}} \cdot W_j^Q \cdot \left( W_j^K \right)^T \right)
          \cdot \left( X^K \right)^T
        + M
      \right)
      \cdot X^V
      \cdot \left( W_j^V \cdot W_j^O \right)
      & &
\end{align*}
$$

Let:

$$
\begin{align*}
  W_j^{(1)} & = \frac{1}{\sqrt{d^K}} \cdot W_j^Q \cdot \left( W_j^K \right)^T & \in \mathbb{R}^{d_m \times d_n} \\
  W_j^{(2)} & = W_j^V \cdot W_j^O & \in \mathbb{R}^{d_n \times d_m} \\
  W_{j, X^K}^{(1)} & = W_j^{(1)} \cdot \left( X^K \right)^T & \in \mathbb{R}^{d_m \times d_{kv}} \\
  W_{j, X^V}^{(2)} & = X^V \cdot W_j^{(2)} & \in \mathbb{R}^{d_{kv} \times d_m} \\
  B^{(1)} & = M & \in \mathbb{R}^{d_q \times d_{kv}}\\
  B^{(2)} & = 0 & \in \mathbb{R}^{d_q \times d_m} \\
  f^{(1)} & = \text{softmax} &
\end{align*}
$$

Then the attention mechanism becomes analogous to a parallelized, fully connected feed-forward network with input-dependent weights, taking the form of equation [2](#eq-2):

<a id="eq-7"></a>

$$
Y_j = f^{(1)} \left( X^Q \cdot W_{j, X^K}^{(1)} + B^{(1)} \right) \cdot W_{j, X^V}^{(2)} + B^{(2)}
  \qquad\quad(7)
$$

Furthermore:

 * Self-attention for $X \in \mathbb{R}^{d_s \times d_m}$ can also be written as:

   <a id="eq-8"></a>

   $$
   Y_j = f^{(1)} \left( X \cdot W_j^{(1)} \cdot X^T + B^{(1)} \right)
       \cdot \left( X \cdot W_j^{(2)} \right) + B^{(2)}
     \qquad\quad(8)
   $$

 * Cross-attention for $X_1 \in \mathbb{R}^{d_1 \times d_m}$ and $X_2 \in \mathbb{R}^{d_2 \times d_n}$ becomes:

   <a id="eq-9"></a>

   $$
   Y_j = f^{(1)} \left( X_1 \cdot W_j^{(1)} \cdot X_2^T + B^{(1)} \right)
       \cdot \left( X_2 \cdot W_j^{(2)} \right) + B^{(2)}
     \qquad\quad(9)
   $$

### 2.1 Notes

 * The idea of comparing the $\text{softmax}$ function in the attention mechanism to the activation function of feed-forward networks, especially the $\max(x, 0)$ function which is also known as $\text{ReLU}$, has been already studied in the literature \[[5](#ref-5), [19](#ref-19)\].

 * $W_j^{(1)}$ and $W_j^{(2)}$ can be thought of as decompositions of some $\hat{W}_j^{(1)}$ and $\hat{W}_j^{(2)}$ weight matrices.

 * $W_{j, X^K}^{(1)}$ and $W_{j, X^V}^{(2)}$ can be thought of as weight matrices generated from the input on-the-fly for a two-layered feed-forward neural network with $B^{(1)}$ and $B^{(2)}$ as bias, making the attention mechanism a hypernetwork \[[6](#ref-6)\] (also known as a fast weight programmer \[[7](#ref-7)\]).

 * The self-attention formula contains a quadratic form, analogous to evaluating the first layer for a second degree polynomial instead of the raw input features. Then the resulting activations are multiplied again by $X$ in the second layer, as if introducing a third degree (aside from the non-linearity). (Cf. kernel trick, polynomial regression, etc.).

 * Similarly, cross-attention can be thought of as a feed-forward network which operates on the product of two inputs.

## 3 Simplifications

The language model interpretation of the scaled dot-product attention does not leave much room for major architectural changes, but the quadratic form or hypernetwork interpretations open the door for multiple ideas.

### 3.1 Replacing softmax with ReLU

The ReLUFormer architecture \[[5](#ref-5)\] is a variation of the Transformer \[[1](#ref-1)\] where the $\text{softmax}$ function in the attention mechanism is replaced with $\text{ReLU}$, and a sequence length dependent scaling factor is introduced in order to control the variance and stabilize the training, along with a regularization term in the loss function which encourages better value weight utilization. (For causal attention, the scaling factor is different for every token.)

There are various strategies and theoretical explanations in the literature for the role of scaling when it comes to replacing $\text{softmax}$ with other functions \[[5](#ref-5), [11](#ref-11), [19](#ref-19)\]. Let's denote scaling factors at various points of this modified, $\text{ReLU}$ attention mechanism by $\gamma_1$ and $\gamma_2$. With these changes, equation [6](#eq-6) becomes:

<a id="eq-10"></a>

$$
Y_j = \gamma_2 \cdot \text{ReLU} \left(
        \gamma_1
        \cdot X^Q \cdot W_j^Q \cdot \left( W_j^K \right)^T \cdot \left( X^K \right)^T
        + M
      \right)
      \cdot X^V \cdot W_j^V \cdot W_j^O
  \qquad\quad(10)
$$

For $\gamma_1$, the traditional choice \[[1](#ref-1)\] is $\gamma_1 = \sqrt{\frac{1}{d^K}}$, and for $\gamma_2$, the recommendation \[[5](#ref-5)\] is $\gamma_2 = \sqrt{\frac{2}{d_{kv}}}$.

(Note: since $\text{ReLU}$ is scalar-multiplicative for positive scalars, and similarly, $\forall 0 < c \in \mathbb{R},\ c \cdot M = M$ due to the construction of $M$, the two scaling factors could be merged into a single one.)

### 3.2 Simplifying the attention computation

#### 3.2.1 Eliminating the K and V projections

With the usual choices of $d^K$ and $d^V$, the $W_j^Q \cdot \left( W_j^K \right)^T \in \mathbb{R}^{d_m \times d_n}$ and the $W_j^V \cdot W_j^O \in \mathbb{R}^{d_n \times d_m}$ matrix products in equation [10](#eq-10) can be viewed as low-rank decompositions of head-specific larger weight matrices.

This poses the question: would it be beneficial to have only a single attention head with potentially full-rank weight matrices, instead of using a sum of $h$ matrices with each having its rank constrained by $d^V$?

The formula of that model would be, for $\hat{W}^Q \in \mathbb{R}^{d_m \times d_n}$ and $\hat{W}^O \in \mathbb{R}^{d_n \times d_m}$ learnable parameters:

<a id="eq-11"></a>

$$
Y = \gamma_2 \cdot \text{ReLU} \left(
      \gamma_1 \cdot X^Q \cdot \hat{W}^Q \cdot \left( X^K \right)^T + M
    \right)
    \cdot X^V \cdot \hat{W}^O
  \qquad\quad(11)
$$

Note: equation [11](#eq-11) has only $2 \cdot d_m \cdot d_n$ parameters while with the usual choices of $d^K$ and $d^V$, equation [10](#eq-10) and the original $\text{softmax}$ attention mechanism both have $d_m^2 + d_n^2 + 2 \cdot d_m \cdot d_n$ parameters.

#### 3.2.2 Eliminating the constant scaling

Scaling of the $\text{softmax}$ function's argument by a constant controls the flatness of the probability distribution (i.e. the entropy) - since there is no longer a probability distribution in the model, the constant scaling should have no noticable effect either (especially since it would be reasonable to expect it to be absorbed by the weight matrices anyways). Thus, $\gamma_1 = 1$ seems to be a reasonable choice.

#### 3.2.3 A computationally cheaper alternative

Due to the calculation of a matrix product of size $d_q \times d_{kv}$ in equation [11](#eq-11), as the sequence lengths grow linearly, the computation complexity grows quadratically.

At the same time, the rank of $X^Q \cdot \hat{W}^Q \cdot \left( X^K \right)^T$ cannot possibly be larger than $\min\{d_m, d_n\}$, and even if the addition of $M$ and the applied non-linearity temporarily establish a greater rank for the $\text{ReLU} \left( X^Q \cdot \hat{W}^Q \cdot \left( X^K \right)^T + M \right)$ matrix (let's denote this matrix by $Z$), the rank of the final product $Y$ is again bounded by $d_n$ due to the multiplication involving $X^V$:

 * It is known from linear algebra that for any $X^V$ matrix, there exists an appropriately sized $G$ matrix called a generalized inverse such that $X^V \cdot G \cdot X^V = X^V$.
 * If $\hat{Z} = Z \cdot X^V \cdot G$, then $\hat{Z} \cdot X^V = Z \cdot X^V$, and due to its construction, $\text{rank}\left( \hat{Z} \right) \le d_n$, therefore for any possible value and rank of $Z$, there exists an equivalent linear mapping with its rank bounded by $d_n$.

If the sequence lengths $d_q$ and $d_{kv}$ are significantly larger than the embedding dimensions $d_m$ and $d_n$, then performing an at most $d_n$ rank linear mapping with a computational complexity dominated by $d_q \cdot d_{kv}$ seems wasteful.

However, if $d_m = d_n$ and $d_q = d_{kv}$ and $M = 0$ (like it is the case in self-attention with no masking), then a computationally cheaper alternative model may be to calculate the dot product of the columns instead of the rows:

<a id="eq-12"></a>

$$
Y = \gamma_2 \cdot X^V
    \cdot \text{ReLU} \left(
      \gamma_1 \cdot \left( X^K \right)^T \cdot X^Q \cdot \hat{W}^Q
    \right)
    \cdot \hat{W}^O
  \qquad\quad(12)
$$

This formula no longer produces a similarity-based weighted sum of tokens - instead, it may be interpreted as computing a projection matrix based on compressed information about the input features.

Equation [12](#eq-12) requires only $\mathcal{O} \left( d_m^2 \cdot d_q + d_m^3 \right)$ operations to produce a $d_m \times d_m$ matrix which then projects a $d_q \times d_m$ matrix with $\mathcal{O} \left( d_m^2 \cdot d_q \right)$ operations, followed by another projection with the same complexity. Thus, computational complexity grows only linearly with $d_q$.

This can be simplified further: since now the expression uses the product of two input matrices to compute a projection matrix for the third one, applying another fixed linear projection seems redundant:

<a id="eq-13"></a>

$$
Y = \gamma_2 \cdot X^V
    \cdot \text{ReLU} \left(
      \gamma_1 \cdot \left( X^K \right)^T \cdot X^Q \cdot \hat{W}^Q
    \right)
  \qquad\quad(13)
$$

Perhaps, a better use of $d_m^2$ model weights would be to use them additively, turning the projection matrix calculator expression into a perceptron with a bias matrix.

For $B \in \mathbb{R}^{d_m \times d_m}$ learnable parameters, let:

<a id="eq-14"></a>

$$
Y = \gamma_2 \cdot X^V
    \cdot \text{ReLU} \left(
      \gamma_1 \cdot \left( X^K \right)^T \cdot X^Q \cdot \hat{W}^Q + B
    \right)
  \qquad\quad(14)
$$

### 3.3 Drop-in

This technique does not simplify the model's static computation formula, instead, a different kind of simplification is done dynamically during training.

Drop-out regularization \[[10](#ref-10)\] reduces overfitting on the training data by preventing complex co-adaptations between neurons, inspired by the idea of genetic recombination forcing genes to encode traits mostly independently from each other. To achieve this, drop-out regularization randomly sets some of the activations of the preceding layer to zero and scales the surviving ones in order to restore the expected magnitude of the changed vector before forwarding it to subsequent layers during training, while during inference, all activations are kept intact. This can be viewed as training several sub-networks simultaneously and having the final model be an ensemble of those.

Aside from drop-out regularization and other similar techniques \[[12](#ref-12)\], artificial neural networks are usually trained with the entire network architecture already in place at the beginning. Drop-in changes that by starting the training only with a small subset of the neurons in each layer, and gradually increasing the model's complexity as the training progresses.

To achieve this, drop-in zeroes out most of the input features and hidden layer activations during the training in a deterministically randomized fashion, forcing the remaining small sub-network to learn a coarse model from the surviving features. Then as the training progresses, more and more input features and hidden dimensions are turned on gradually, allowing the network to refine what the smaller, simpler sub-network have learned so far.

Drop-in can be performed alongside traditional drop-out regularization.

## 4 Experiments

The goal of these small experiments is to compare the performance of similarly sized self-attention model variants rather than to train state-of-the-art models.

3 different, fixed random seeds are chosen and each experiment is performed 3 times, once with each random seed, in 2 flavors:

 * with only drop-out ($p = 0.1$),
 * and with drop-in along with drop-out ($p = 0.1$).

For each dataset, the entire set of training samples is considered 1 epoch.

Language models use simple word-based tokenization with a minimum word occurrence of 2. 

The mean performance, the standard deviation, and the model size (the number of learnable parameters) are reported for each experiment and model variant. The model size does not include the state used by the drop-in layers and the text token embedding and output decoding layers (where applicable).

All models use sinusoidal positional encoding.

All experiments were run with AdamW optimizer \[[18](#ref-18)\] $\left( \beta_1 = 0.9,\ \beta_2 = 0.999,\ \text{weight decay} = 0.01 \right)$ and a constant learning rate.

### 4.1 Model variants

All model variants use $\text{ReLU}$ for the activation function of the fully connected feed-forward network blocks, and all normalize before the attention block (pre-LN).

 * **Softmax**: traditional multi-head Transformer Encoder (equation [3](#eq-3) and [4](#eq-4)).
 * **ReLU**: Transformer Encoder with the $\text{softmax}$ function replaced with $\text{ReLU}$ (equation [10](#eq-10)), with $\gamma_1 = \sqrt{\frac{1}{d^K}}$, and $\gamma_2 = 1$.
 * **SReLU**: the single-head, simplified version of the **ReLU** variant, from equation [11](#eq-11).
 * **SReLU/noscl**: same as **SReLU**, but with $\gamma_1 = \gamma_2 = 1$.
 * **UbPerc**: the model described by equation [13](#eq-13) with $\gamma_1 = \gamma_2 = 1$.
 * **Perc**: the model described by equation [14](#eq-14) with $\gamma_1 = \gamma_2 = 1$.

The experiments where the input sequence length varies are also run with model variants which set $\gamma_1 = 1$ and $\gamma_2 = \sqrt{\frac{2}{d_{kv}}}$. These are denoted by the **/seql** suffix.

All model variants share the same hyperparameters except for the number of attention heads ($h$) which is set to 1 for the **SReLU**, **UbPerc**, and the **Perc** variants.

### 4.2 Drop-in schedule

The training begins with only 20% of the input features and neurons being active. As the training progresses, the active ratio is increased to 100% in discrete steps. The active ratio is left unchanged for a while at the beginning and at the end of the training.

### 4.3 Classification tasks

 * The models are trained directly on the classification task from scratch, without any pre-training or transfer learning.
 * In the text classification tasks, the samples in each batch are padded to match the length of the longest sample in that batch. The padding tokens are masked, except for the UbPerc and Perc models which don't support masking.
 * The reported metric is the $F_1$ score achieved on the test set, in order to comprise both precision and recall. (Higher scores are better.)

#### 4.3.1 AG's News

 * **Task**: classification of news headlines and lead texts into 4 categories, with 120 000 training samples (5% held out for validation) and 7600 test samples, both sets balanced \[[13](#ref-13)\].
 * **Model**:
    * Layers: 6.
    * Classification head: mean pooling followed by a single linear projection.
    * Embedding dimensions: 72.
    * MLP dimensions: 288.
    * Attention heads: 3.
 * **Training**:
    * Epochs: 1.
    * Batch size: 8.
    * Gradient accumulation: 4 batches.
    * Learning rate: 0.0005.
    * Loss function: cross-entropy.

#### 4.3.2 CIFAR-10

 * **Task**: classification of 32x32 pixel RGB images into 10 classes, with 50 000 training samples (5% held out for validation) and 10 000 test samples, both sets balanced \[[14](#ref-14)\].
 * **Model**: Vision Transformer \[[2](#ref-2)\].
    * Patch size: 4x4.
    * Layers: 12.
    * Classification head: single linear projection of a manually inserted classification token.
    * Embedding dimensions: 72.
    * MLP dimensions: 288.
    * Attention heads: 3.
 * **Training**:
    * Epochs: 100.
    * Augmentation: affine transformations, color temperature, sharpening, blurring, noise, etc.
    * Batch size: 128.
    * Gradient accumulation: 4 batches.
    * Learning rate: 0.001.
    * Loss function: cross-entropy.

#### 4.3.3 CIFAR-100

 * **Task**: classification of 32x32 pixel RGB images into 20 coarse classes, with 50 000 training samples (5% held out for validation) and 10 000 test samples, both sets balanced \[[14](#ref-14)\].
 * **Model**: Vision Transformer \[[2](#ref-2)\].
    * Patch size: 4x4.
    * Layers: 12.
    * Classification head: single linear projection of a manually inserted classification token.
    * Embedding dimensions: 128.
    * MLP dimensions: 512.
    * Attention heads: 4.
 * **Training**:
    * Epochs: 50.
    * Augmentation: affine transformations, color temperature, sharpening, blurring, noise, etc.
    * Batch size: 128.
    * Gradient accumulation: 4 batches.
    * Learning rate: 0.001.
    * Loss function: cross-entropy.

#### 4.3.4 IMDb

 * **Task**: classification of movie review texts written by IMDb users, into positive and negative sentiment, with 25 000 training samples (5% held out for validation) and 25 000 test samples, both sets balanced \[[15](#ref-15)\]. The unlabeled samples in the dataset and a small subset of the samples which are longer than 4500 characters were discarded.
 * **Model**:
    * Layers: 9.
    * Classification head: mean pooling followed by a single linear projection.
    * Embedding dimensions: 144.
    * MLP dimensions: 576.
    * Attention heads: 6.
 * **Training**:
    * Epochs: 1.
    * Batch size: 8.
    * Gradient accumulation: 4 batches.
    * Learning rate: 0.0005.
    * Loss function: binary cross-entropy.

#### 4.3.5 ListOps

 * **Task**: evaluating prefix-notation number lists and mathematical list operations (e.g. `MIN`, `MAX`, `SUM`, etc.) modulo 10, i.e. classification of formulas with a strict syntax into 10 classes. The training and the test set contain 90 000 and 10 000 samples respectively, mostly balanced \[[16](#ref-16)\] (5% of the training set is held out for validation). A small subset of the samples were discarded which are longer than 1000 characters.
 * **Model**:
    * Layers: 12.
    * Classification head: mean pooling followed by a single linear projection.
    * Embedding dimensions: 144.
    * MLP dimensions: 576.
    * Attention heads: 6.
 * **Training**:
    * Epochs: 2.
    * Batch size: 2.
    * Gradient accumulation: 16 batches.
    * Learning rate: 0.0005.
    * Loss function: cross-entropy.

### 4.4 Text generation

The objective is next-token prediction using self-attention with causal masking. Since the UbPerc and Prec models don't support masking, they are omitted from these experiments. The reported metric is the preplexity measured on the test set. (Lower score is better.)

#### 4.4.1 IMDb-gen

 * **Task**: next-token prediction in movie review texts written by IMDb users \[[15](#ref-15)\]. The training set is comprised of the 25 000 labeled training samples and the 50 000 unlabeled samples (5% of the training set is held out for validation), the test set is the 25 000 test samples from the dataset.
 * **Model**:
    * Layers: 12.
    * Sequence length: 256.
    * Embedding dimensions: 256.
    * MLP dimensions: 1024.
    * Attention heads: 8.
 * **Training**:
    * Epochs: 1.
    * Batch size: 8.
    * Gradient accumulation: 32 batches.
    * Learning rate: 0.0005.
    * Loss function: cross-entropy.

#### 4.4.2 WikiText2

 * **Task**: next-token prediction in a selection of Wikipedia articles \[[17](#ref-17)\]. The training, validation, and the test sets contain approximately 10 MB, 1.1 MB, and 1.2 MB of English text respectively.
 * **Model**:
    * Layers: 12.
    * Sequence length: 256.
    * Embedding dimensions: 192.
    * MLP dimensions: 768.
    * Attention heads: 6.
 * **Training**:
    * Epochs: 20.
    * Batch size: 8.
    * Gradient accumulation: 32 batches.
    * Learning rate: 0.0005.
    * Loss function: cross-entropy.

## 5 Results

The following table shows the experiment results in `mean ± SD (model size)` format on the test set. (The model size does not include text embedding and text output token decoding layers.) The best result of each experiment is highlighted with bold, the second best one is underlined.

TBD

### 5.1 Discussion

TBD

## 6 Future work

 * Do the presented results scale to wider, deeper models, larger datasets, and longer sequences?
 * How do the presented techniques play along with:
    * other activation functions than $\text{ReLU}$,
    * other transformer optimization techniques,
    * stochastic depth \[[12](#ref-12)\],
    * recurrence?
 * Is there an optimal scheduling for drop-in?
 * Does dynamically changing drop-out ratio during training have similar effects to drop-in, e.g. when done in a similar fashion?

## References

1. <a id="ref-1"></a> Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, and Illia Polosukhin, "*Attention Is All You Need*" (2017). DOI: [https://doi.org/10.48550/arXiv.1706.03762](https://doi.org/10.48550/arXiv.1706.03762)

2. <a id="ref-2"></a> Alexey Dosovitskiy, Lucas Beyer, Alexander Kolesnikov, Dirk Weissenborn, Xiaohua Zhai, Thomas Unterthiner, Mostafa Dehghani, Matthias Minderer, Georg Heigold, Sylvain Gelly, Jakob Uszkoreit,  and Neil Houlsby, "*An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale*" (2021). DOI: [https://doi.org/10.48550/arXiv.2010.11929](https://doi.org/10.48550/arXiv.2010.11929)

3. <a id="ref-3"></a> Simon Rouard, Francisco Massa, and Alexandre Défossez, "*Hybrid Transformers for Music Source Separation*" (2022). DOI: [https://doi.org/10.48550/arXiv.2211.08553](https://doi.org/10.48550/arXiv.2211.08553)

4. <a id="ref-4"></a> Kaiming He, Xiangyu Zhang, Shaoqing Ren, and Jian Sun, "*Deep Residual Learning for Image Recognition*" (2015). DOI: [https://doi.org/10.48550/arXiv.1512.03385](https://doi.org/10.48550/arXiv.1512.03385)

5. <a id="ref-5"></a>Kai Shen, Junliang Guo, Xu Tan, Siliang Tang, Rui Wang, and Jiang Bian, "*A Study on ReLU and Softmax in Transformer*" (2023). DOI: [https://doi.org/10.48550/arXiv.2302.06461](https://doi.org/10.48550/arXiv.2302.06461)

6. <a id="ref-6"></a> Simon Schug, Seijin Kobayashi, Yassir Akram, João Sacramento, and Razvan Pascanu, "*Attention as a Hypernetwork*" (2025). DOI: [https://doi.org/10.48550/arXiv.2406.05816](https://doi.org/10.48550/arXiv.2406.05816)

7. <a id="ref-7"></a> Imanol Schlag, Kazuki Irie, and Jürgen Schmidhuber, "*Linear Transformers Are Secretly Fast Weight Programmers*" (2021). DOI: [https://doi.org/10.48550/arXiv.2102.11174](https://doi.org/10.48550/arXiv.2102.11174)

8. <a id="ref-8"></a> Jimmy Lei Ba, Jamie Ryan Kiros, and Geoffrey E. Hinton, "*Layer Normalization*" (2016). DOI: [https://doi.org/10.48550/arXiv.1607.06450](https://doi.org/10.48550/arXiv.1607.06450)

9. <a id="ref-9"></a> Ruibin Xiong, Yunchang Yang, Di He, Kai Zheng, Shuxin Zheng, Chen Xing, Huishuai Zhang, Yanyan Lan, Liwei Wang, and Tie-Yan Liu, "*On Layer Normalization in the Transformer Architecture*" (2020). DOI: [https://doi.org/10.48550/arXiv.2002.04745](https://doi.org/10.48550/arXiv.2002.04745)

10. <a id="ref-10"></a> Nitish Srivastava, Geoffrey Hinton, Alex Krizhevsky, Ilya Sutskever, and Ruslan Salakhutdinov, "*Dropout: A Simple Way to Prevent Neural Networks from Overfitting*" (2014). URL: [http://jmlr.org/papers/v15/srivastava14a.html](http://jmlr.org/papers/v15/srivastava14a.html)

11. <a id="ref-11"></a> Hemanth Saratchandran, Jianqiao Zheng, Yiping Ji, Wenbo Zhang, and Simon Lucey, "*Rethinking Attention: Polynomial Alternatives to Softmax in Transformers*" (2024). DOI: [https://doi.org/10.48550/arXiv.2410.18613](https://doi.org/10.48550/arXiv.2410.18613)

12. <a id="ref-12"></a> Gao Huang, Yu Sun, Zhuang Liu, Daniel Sedra, and Kilian Weinberger, "*Deep Networks with Stochastic Depth*" (2016). DOI: [https://doi.org/10.48550/arXiv.1603.09382](https://doi.org/10.48550/arXiv.1603.09382)

13. <a id="ref-13"></a> Xiang Zhang, Junbo Jake Zhao, and Yann LeCun, "*Character-level Convolutional Networks for Text Classification*" (2015). DOI: [https://doi.org/10.48550/arXiv.1509.01626](https://doi.org/10.48550/arXiv.1509.01626)

14. <a id="ref-14"></a> Alex Krizhevsky, "*Learning Multiple Layers of Features from Tiny Images*" (2009). URL: [https://cave.cs.toronto.edu/kriz/learning-features-2009-TR.pdf](https://cave.cs.toronto.edu/kriz/learning-features-2009-TR.pdf)

15. <a id="ref-15"></a> Andrew L. Maas, Raymond E. Daly, Peter T. Pham, Dan Huang, Andrew Y. Ng, and  Christopher Potts "*Learning Word Vectors for Sentiment Analysis*" (2011). URL: [https://aclanthology.org/P11-1015/](https://aclanthology.org/P11-1015/)

16. <a id="ref-16"></a> Nikita Nangia and Samuel R. Bowman, "*ListOps: A Diagnostic Dataset for Latent Tree Learning*" (2018). DOI: [https://doi.org/10.48550/arXiv.1804.06028](https://doi.org/10.48550/arXiv.1804.06028)

17. <a id="ref-17"></a> Stephen Merity, Caiming Xiong, James Bradbury, and Richard Socher, "*Pointer Sentinel Mixture Models*" (2016). DOI: [https://doi.org/10.48550/arXiv.1609.07843](https://doi.org/10.48550/arXiv.1609.07843)

18. <a id="ref-18"></a> Ilya Loshchilov and Frank Hutter, "*Decoupled Weight Decay Regularization*" (2019). DOI: [https://doi.org/10.48550/arXiv.1711.05101](https://doi.org/10.48550/arXiv.1711.05101)

19. <a id="ref-19"></a>Mitchell Wortsman, Jaehoon Lee, Justin Gilmer, and Simon Kornblith, "*Replacing softmax with ReLU in Vision Transformers*" (2023). DOI: [https://doi.org/10.48550/arXiv.2309.08586](https://doi.org/10.48550/arXiv.2309.08586)

## License

[CC-BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/)

**You are free to:**

 * **Share** - copy and redistribute the material in any medium or format for any purpose, even commercially.
 * **Adapt** - remix, transform, and build upon the material for any purpose, even commercially.

The licensor cannot revoke these freedoms as long as you follow the license terms.

**Under the following terms:**

 * **Attribution** - You must give [appropriate credit](https://creativecommons.org/licenses/by-sa/4.0/#ref-appropriate-credit), provide a link to the license, and [indicate if changes were made](https://creativecommons.org/licenses/by-sa/4.0/#ref-indicate-changes). You may do so in any reasonable manner, but not in any way that suggests the licensor endorses you or your use.
 * **ShareAlike** - If you remix, transform, or build upon the material, you must distribute your contributions under the [same license](https://creativecommons.org/licenses/by-sa/4.0/#ref-same-license) as the original.

**No additional restrictions** - You may not apply legal terms or [technological measures](https://creativecommons.org/licenses/by-sa/4.0/#ref-technological-measures) that legally restrict others from doing anything the license permits.

**Notices:**

 * You do not have to comply with the license for elements of the material in the public domain or where your use is permitted by an applicable [exception or limitation](https://creativecommons.org/licenses/by-sa/4.0/#ref-exception-or-limitation).

 * No warranties are given. The license may not give you all of the permissions necessary for your intended use. For example, other rights such as [publicity, privacy, or moral rights](https://creativecommons.org/licenses/by-sa/4.0/#ref-publicity-privacy-or-moral-rights) may limit how you use the material.

## Appendix: Code

### Dependencies

In [1]:
# !pip install "mlx"                        # Apple Silicon
# !pip install "mlx[cuda]"                  # CUDA
# !pip install "mlx[cpu]"                   # CPU

# !pip install "matplotlib" "numpy" "pillow" "scikit-learn"

In [2]:
import collections
import collections.abc
import csv
import datetime
import enum
import gzip
import hashlib
import json
import math
import os
import pathlib
import random
import re
import tarfile
import tempfile
import time
import typing
import urllib.request

import matplotlib.pyplot as plt

import mlx.core
import mlx.nn
import mlx.optimizers
import mlx.utils

import numpy as np

import PIL.Image

import sklearn.metrics

### Default device

In [3]:
print(mlx.core.default_device())

Device(gpu, 0)


### Paths

In [4]:
DATA_PATH = pathlib.Path("./data")
MODELS_PATH = pathlib.Path("./models")
RESULTS_PATH = pathlib.Path("./results")

### Log

In [5]:
def log_msg(msg: str, log_path: typing.Optional[pathlib.Path]=None):
    print(msg)

    if log_path is None:
        return

    if not log_path.exists():
        log_path.parent.mkdir(parents=True, exist_ok=True)

    with open(str(log_path), "a") as f:
        print(msg, file=f)

### Debug

In [6]:
DEBUG = False

### Random seed

In [7]:
def set_random_seed(
        seed: int,
        log_path: typing.Optional[pathlib.Path]=None,
):
    log_msg(f"  Random seed: {seed}", log_path=log_path)
    random.seed(seed)
    np.random.seed(seed)
    mlx.core.random.seed(seed)

### Model components

#### Attention

In [8]:
class AttentionType(str, enum.Enum):
    SOFTMAX = "softmax"
    RELU = "relu"
    SIMPLIFIED_RELU = "simplified-relu"
    UNBIASED_PERCEPTRON = "perceptron-unbiased"
    PERCEPTRON = "perceptron"


class AttentionScaling(str, enum.Enum):
    NONE = "none"
    DIMS = "dims"
    SEQ_LEN = "seq_len"


class MultiHeadAttention(mlx.nn.Module):
    def __init__(
            self,
            h: int,
            d_m: int,
            d_n: typing.Optional[int]=None,
            d_K: typing.Optional[int]=None,
            d_V: typing.Optional[int]=None,
            d_ffn: typing.Optional[int]=None,
            attention_type: AttentionType=AttentionType.SOFTMAX,
            scaling: AttentionScaling=AttentionScaling.DIMS,
    ):
        super().__init__()

        d_n = d_n if d_n is not None else d_m
        d_ffn = d_ffn if d_ffn is not None else (4 * d_m)

        if d_K is None:
            if d_m % h != 0:
                raise ValueError(
                    f"Query dimension must be divisable by the number of heads, got {d_m=}, {h=}"
                )

            d_K = d_m // h

        if d_V is None:
            if d_n % h != 0:
                raise ValueError(
                    f"Value dimension must be divisable by the number of heads, got {d_n=}, {h=}"
                )

            d_V = d_n // h

        self.d_m = d_m
        self.d_n = d_n
        self.h = h
        self.d_K = d_K
        self.d_V = d_V
        self.const_scale = (
            (1.0 / math.sqrt(self.d_K))
            if scaling == AttentionScaling.DIMS
            else 1.0
        )
        self.needs_seq_len_scaling = scaling == AttentionScaling.SEQ_LEN

        self.attention_fn = (
            (lambda Y: mlx.core.softmax(Y, axis=-1))
            if attention_type == AttentionType.SOFTMAX
            else lambda Y: mlx.core.maximum(Y, 0.0)
        )

        self.W_Q = None
        self.W_K = None
        self.W_V = None
        self.W_O = None
        self.B = None

        if (
                attention_type == AttentionType.SOFTMAX
                or attention_type == AttentionType.RELU
        ):
            self.compute_fn = self.compute_traditional
            self.W_Q = mlx.nn.Linear(input_dims=d_m, output_dims=h * d_K, bias=False)
            self.W_K = mlx.nn.Linear(input_dims=d_n, output_dims=h * d_K, bias=False)
            self.W_V = mlx.nn.Linear(input_dims=d_n, output_dims=h * d_V, bias=False)
            self.W_O = mlx.nn.Linear(input_dims=h * d_V, output_dims=d_m, bias=False)
        else:
            if h != 1:
                raise ValueError(
                    f"The number of heads must be 1 for {attention_type=!r}, got {h=}"
                )

            if attention_type == AttentionType.SIMPLIFIED_RELU:
                self.compute_fn = self.compute_simplified
                self.W_Q = mlx.nn.Linear(input_dims=d_m, output_dims=d_n, bias=False)
                self.W_O = mlx.nn.Linear(input_dims=d_n, output_dims=d_m, bias=False)
            else:
                if d_m != d_n:
                    raise ValueError(
                        f"The query and key-value dimensions must be equal for {attention_type=!r}, got {dm_=}, {d_n=}"
                    )
    
                if attention_type == AttentionType.UNBIASED_PERCEPTRON:
                    self.compute_fn = self.compute_perceptron
                    self.W_Q = mlx.nn.Linear(input_dims=d_m, output_dims=d_m, bias=False)
        
                elif attention_type == AttentionType.PERCEPTRON:
                    self.compute_fn = self.compute_perceptron
                    rnd_range = math.sqrt(1.0 / d_m)
                    self.W_Q = mlx.nn.Linear(input_dims=d_m, output_dims=d_m, bias=False)
                    self.B = mlx.core.random.uniform(
                        low=-rnd_range,
                        high=rnd_range,
                        shape=(d_m, d_m),
                    )
        
                else:
                    raise ValueError(
                        "attention_type must be one of "
                        + ", ".join(str(a) for a in AttentionType)
                        + f"; got {attention_type!r}"
                    )

    def __call__(
            self,
            X_Q: mlx.core.array,
            X_K: mlx.core.array,
            X_V: mlx.core.array,
            M: typing.Optional[mlx.core.array]=None,
            seq_len: typing.Optional[typing.Union[int, float, mlx.core.array]]=None,
    ) -> mlx.core.array:
        return self.compute_fn(
            X_Q=X_Q,
            X_K=X_K,
            X_V=X_V,
            seq_len=seq_len if seq_len is not None else float(X_V.shape[1]),
            M=M,
        )

    @staticmethod
    def _scale_by_seq_len(
            Y: mlx.core.array,
            seq_len: typing.Union[int, float, mlx.core.array]
    ) -> mlx.core.array:
        scale = mlx.core.sqrt(seq_len / 2.0) + 1e-6
        scale = scale.reshape(scale.shape + (1,) * (Y.ndim - scale.ndim))

        return Y / scale 

    def compute_traditional(
            self,
            X_Q: mlx.core.array,
            X_K: mlx.core.array,
            X_V: mlx.core.array,
            seq_len: typing.Union[int, float, mlx.core.array],
            M: typing.Optional[mlx.core.array]=None,
    ) -> mlx.core.array:
        # X_Q shape: [b, d_q, d_m]
        # X_K, X_V shape: [b, d_kv, d_n]

        Q = self.W_Q(X_Q) # Shape: [b, d_q, h * d_K]
        K = self.W_K(X_K) # Shape: [b, d_kv, h * d_K]
        V = self.W_V(X_V) # Shape: [b, d_kv, h * d_V]

        # Splitting heads; shapes: [b, d_len, h, d_head]
        Q = mlx.core.unflatten(Q, axis=2, shape=(self.h, self.d_K))
        K = mlx.core.unflatten(K, axis=2, shape=(self.h, self.d_K))
        V = mlx.core.unflatten(V, axis=2, shape=(self.h, self.d_V))

        # Rearrange: [b, d_len, h, d_head] --> [b, h, d_len, d_head]
        Q = mlx.core.transpose(Q, axes=(0, 2, 1, 3))
        V = mlx.core.transpose(V, axes=(0, 2, 1, 3))

        # Rearrange: [b, d_len, h, d_head] --> [b, h, d_K, d_kv]
        K_T = mlx.core.transpose(K, axes=(0, 2, 3, 1))

        Y = self.const_scale * (Q @ K_T) # Shape: [b, h, d_q, d_kv]

        if M is not None:
            Y = Y + M

        # Shape: [b, h, d_q, d_V]
        Y = self.attention_fn(Y) @ V

        # Rearrange: [b, h, d_q, d_V] --> [b, d_q, h, d_V]
        Y = mlx.core.transpose(Y, axes=(0, 2, 1, 3))

        if self.needs_seq_len_scaling:
            Y = self._scale_by_seq_len(Y, seq_len)

        # Concatenate: [b, d_q, h, d_V] --> [b, d_q, h * d_V]
        Y = mlx.core.flatten(Y, start_axis=2, end_axis=3)

        Y = self.W_O(Y) # Shape: [b, d_q, d_m]

        return Y

    def compute_simplified(
            self,
            X_Q: mlx.core.array,
            X_K: mlx.core.array,
            X_V: mlx.core.array,
            seq_len: typing.Union[int, float, mlx.core.array],
            M: typing.Optional[mlx.core.array]=None,
    ) -> mlx.core.array:
        # X_Q shape: [b, d_q, d_m]
        # X_K, X_V shape: [b, d_kv, d_n]

        Q = self.W_Q(X_Q)        # Shape: [b, d_q, d_n]
        K = X_K                  # Shape: [b, d_kv, d_n]
        V = X_V                  # Shape: [b, d_kv, d_n]

        # Rearrange: [b, d_kv, d_n] --> [b, d_n, d_kv]
        K_T = mlx.core.transpose(K, axes=(0, 2, 1))

        Y = self.const_scale * (Q @ K_T) # Shape: [b, d_q, d_kv]

        if M is not None:
            Y = Y[..., None, :, :] + M
            Y = mlx.core.flatten(Y, start_axis=1, end_axis=2)

        # Shape: [b, d_q, d_n]
        Y = self.attention_fn(Y) @ V

        if self.needs_seq_len_scaling:
            Y = self._scale_by_seq_len(Y, seq_len)

        Y = self.W_O(Y) # Shape: [b, d_q, d_m]

        return Y

    def compute_perceptron(
            self,
            X_Q: mlx.core.array,
            X_K: mlx.core.array,
            X_V: mlx.core.array,
            seq_len: typing.Union[int, float, mlx.core.array],
            M: typing.Optional[mlx.core.array]=None,
    ) -> mlx.core.array:
        # X_Q shape: [b, d_q, d_m]
        # X_K, X_V shape: [b, d_kv, d_m]

        d_q = X_Q.shape[-2]
        d_kv = X_K.shape[-2]

        assert d_q == d_kv, f"Key and Query sequence lengths must match, got {d_q=}, {d_kv=}"

        seq_len_col = (
            mlx.core.array([[seq_len]])
            if isinstance(seq_len, (int, float))
            else seq_len[:, None]
        )
        col_idx = mlx.core.arange(d_q)
        seq_mask = (col_idx < seq_len_col).astype(mlx.core.float32)[..., None]

        Q = self.W_Q(X_Q) * seq_mask   # Shape: [b, d_q, d_n]
        K = X_K * seq_mask             # Shape: [b, d_q, d_m]
        V = X_V * seq_mask             # Shape: [b, d_q, d_m]

        # Rearrange: [b, d_q, d_m] --> [b, d_m, d_q]
        K_T = mlx.core.transpose(K, axes=(0, 2, 1))

        Y = self.const_scale * (K_T @ Q) # Shape: [b, d_m, d_m]

        if self.B is not None:
            Y = Y + self.B

        # No additive masking!

        # Shape: [b, d_q, d_m]
        Y = V @ self.attention_fn(Y)

        if self.needs_seq_len_scaling:
            Y = self._scale_by_seq_len(Y, seq_len)

        return Y

#### Drop-in

In [9]:
class DropIn(mlx.nn.Dropout):
    @staticmethod
    def set_dims(module: mlx.nn.Module, dims: typing.Union[int, float]):
        for name, sub_module in module.named_modules():
            if not isinstance(sub_module, DropIn):
                continue

            keep_dims = (
                dims
                if isinstance(dims, int)
                else int(sub_module.max_dims * dims + 0.5)
            )
            assert 0 < keep_dims <= sub_module.max_dims, \
                f"Sub-module {name!r} cannot have {dims} dimensions kept ({sub_module.max_dims=})"

            sub_module.dims = keep_dims

    def __init__(self, max_dims: int, p: float=0.5):
        super().__init__(p=p)

        assert 0 < max_dims
        assert 0.0 <= p < 1.0

        self.dims = max_dims
        self.max_dims = max_dims
        self.indices = mlx.core.random.permutation(
            mlx.core.array(list(range(max_dims)))
        )
        self.freeze(keys=["indices"])

    def _extra_repr(self) -> str:
        return f"max_dims={self.max_dims}, " + super()._extra_repr()

    def unfreeze(self, *args, **kwargs):
        super().unfreeze(*args, **kwargs)

        self.freeze(keys=["indices"])

    def __call__(self, x: mlx.core.array) -> mlx.core.array:
        if not self.training:
            return x

        scale = self.max_dims / self.dims
        zero_idx = self.indices[self.dims:]
        mask = mlx.core.full(self.max_dims, vals=scale)
        mask[zero_idx] = 0.0

        return super().__call__(x * mask)


def test_drop_in():
    x = mlx.core.array([list(range(1, 101)), list(range(101, 201))])

    drop_in_99p = DropIn(max_dims=100, p=0.99999999)
    drop_in_99p.eval()
    y_1 = drop_in_99p(x)
    drop_in_99p.train()
    y_2 = drop_in_99p(x)

    assert (y_1 != x).sum().item() == 0
    assert (y_2 != 0.0).sum().item() == 0

    drop_in_0p = DropIn(max_dims=100, p=0.0)
    seq = mlx.nn.Sequential(drop_in_0p)
    DropIn.set_dims(seq, 50)
    drop_in_0p.eval()
    y_1 = drop_in_0p(x)
    drop_in_0p.train()
    y_2 = drop_in_0p(x)

    assert (y_1 != x).sum().item() == 0
    assert (y_2 == 0.0).sum().item() == x.shape[0] * 50

    weights_file_fd, weights_file_name = tempfile.mkstemp(
        prefix="drop-in-test-",
        suffix=".safetensors",
    )
    os.close(weights_file_fd)

    try:
        drop_in_0p.save_weights(weights_file_name)
        drop_in_restored = DropIn(max_dims=100, p=0.0)
        drop_in_restored.load_weights(weights_file_name)
        DropIn.set_dims(drop_in_restored, 50)
        drop_in_restored.train()
        y_3 = drop_in_restored(x)

    finally:
        os.unlink(weights_file_name)

    assert (y_3 != y_2).sum().item() == 0
    
    drop_in_50p = DropIn(max_dims=100, p=0.5)
    DropIn.set_dims(drop_in_50p, 50)
    drop_in_50p.eval()
    y_1 = drop_in_50p(x)
    drop_in_50p.train()
    y_2 = drop_in_50p(x)
    DropIn.set_dims(seq, 0.5)
    y_3 = drop_in_50p(x)

    assert (y_1 != x).sum().item() == 0
    assert 65 <= (y_2 == 0.0).sum().item() <= x.shape[0] * 85
    assert 65 <= (y_3 == 0.0).sum().item() <= x.shape[0] * 85


test_drop_in()

#### Transformer

In [10]:
def make_causal_mask(
        shape: collections.abc.Sequence[int],
        dtype: mlx.core.Dtype=mlx.core.float32,
) -> mlx.core.array:
    return mlx.core.triu(
        x=mlx.core.full(
            shape=shape, vals=-mlx.core.inf, dtype=dtype
        ),
        k=1,
    )


def make_padding_mask(
        is_padding: mlx.core.array,
        dtype: mlx.core.Dtype=mlx.core.float32,
) -> mlx.core.array:
    return mlx.core.where(
        is_padding[:, None, None, :], -mlx.core.inf, 0.0
    ).astype(dtype)


class Transformer(mlx.nn.Module):
    def __init__(
            self,
            n_encoders: int,
            n_decoders: int,
            h: int,
            d_m: int,
            d_ffn: typing.Optional[int]=None,
            dropout_p: float=0.1,
            attention_type: AttentionType=AttentionType.SOFTMAX,
            scaling: AttentionScaling=AttentionScaling.DIMS,
    ):
        super().__init__()

        self.encoder = TransformerEncoder(
            n=n_encoders,
            h=h,
            d_m=d_m,
            d_ffn=d_ffn,
            dropout_p=dropout_p,
            attention_type=attention_type,
            scaling=scaling,
        )
        self.decoder = TransformerDecoder(
            n=n_decoders,
            h=h,
            d_m=d_m,
            d_ffn=d_ffn,
            dropout_p=dropout_p,
            attention_type=attention_type,
            scaling=scaling,
        )

    def __call__(
            self,
            X_Q: mlx.core.array,
            X_KV: mlx.core.array,
            M_Q: typing.Optional[mlx.core.array]=None,
            M_KV: typing.Optional[mlx.core.array]=None,
            seq_len_Q: typing.Optional[typing.Union[float, mlx.core.array]]=None,
            seq_len_KV: typing.Optional[typing.Union[float, mlx.core.array]]=None,
    ) -> mlx.core.array:
        X_KV = self.encoder(X=X_KV, M=M_KV, seq_len=seq_len_KV)
        X_Q = self.decoder(
            X_Q=X_Q,
            X_KV=X_KV,
            M_Q=M_Q,
            M_KV=M_KV,
            seq_len_Q=seq_len_Q,
            seq_len_KV=seq_len_KV,
        )

        return X_Q


class TransformerEncoder(mlx.nn.Module):
    def __init__(
            self,
            n: int,
            h: int,
            d_m: int,
            d_ffn: typing.Optional[int]=None,
            dropout_p: float=0.1,
            attention_type: AttentionType=AttentionType.SOFTMAX,
            scaling: AttentionScaling=AttentionScaling.DIMS,
    ):
        super().__init__()

        self.layers = [
            TransformerEncoderLayer(
                h=h,
                d_m=d_m,
                d_ffn=d_ffn,
                dropout_p=dropout_p,
                attention_type=attention_type,
                scaling=scaling,
            )
            for _ in range(n)
        ]
        self.layer_norm = mlx.nn.LayerNorm(dims=d_m)

    def __call__(
            self,
            X: mlx.core.array,
            M: typing.Optional[mlx.core.array]=None,
            seq_len: typing.Optional[typing.Union[float, mlx.core.array]]=None,
    ) -> mlx.core.array:
        for layer in self.layers:
            X = layer(X, M=M, seq_len=seq_len)

        return self.layer_norm(X)


class TransformerDecoder(mlx.nn.Module):
    def __init__(
            self,
            n: int,
            h: int,
            d_m: int,
            d_ffn: typing.Optional[int]=None,
            dropout_p: float=0.1,
            attention_type: AttentionType=AttentionType.SOFTMAX,
            scaling: AttentionScaling=AttentionScaling.DIMS,
    ):
        super().__init__()

        self.layers = [
            TransformerDecoderLayer(
                h=h,
                d_m=d_m,
                d_ffn=d_ffn,
                dropout_p=dropout_p,
                attention_type=attention_type,
                scaling=scaling,
            )
            for _ in range(n)
        ]
        self.layer_norm = mlx.nn.LayerNorm(dims=d_m)

    def __call__(
            self,
            X_Q: mlx.core.array,
            X_KV: mlx.core.array,
            M_Q: typing.Optional[mlx.core.array]=None,
            M_KV: typing.Optional[mlx.core.array]=None,
            seq_len_Q: typing.Optional[typing.Union[float, mlx.core.array]]=None,
            seq_len_KV: typing.Optional[typing.Union[float, mlx.core.array]]=None,
    ) -> mlx.core.array:
        for layer in self.layers:
            X_Q = layer(
                X_Q=X_Q,
                X_KV=X_KV,
                M_Q=M_Q,
                M_KV=M_KV,
                seq_len_Q=seq_len_Q,
                seq_len_KV=seq_len_KV,
            )

        return self.layer_norm(X_Q)


class TransformerEncoderLayer(mlx.nn.Module):
    def __init__(
            self,
            h: int,
            d_m: int,
            d_ffn: typing.Optional[int]=None,
            dropout_p: float=0.1,
            attention_type: AttentionType=AttentionType.SOFTMAX,
            scaling: AttentionScaling=AttentionScaling.DIMS,
    ):
        super().__init__()

        d_ffn = d_ffn if d_ffn is not None else 4 * d_m

        self.layer_norm = mlx.nn.LayerNorm(dims=d_m)
        self.attention = MultiHeadAttention(
            h=h,
            d_m=d_m,
            attention_type=attention_type,
            scaling=scaling,
        )
        self.dropout = DropIn(max_dims=d_m, p=dropout_p)
        self.fcffn = mlx.nn.Sequential(
            mlx.nn.LayerNorm(dims=d_m),
            mlx.nn.Linear(input_dims=d_m, output_dims=d_ffn),
            mlx.nn.ReLU(),
            DropIn(max_dims=d_ffn, p=dropout_p),
            mlx.nn.Linear(input_dims=d_ffn, output_dims=d_m),
            DropIn(max_dims=d_m, p=dropout_p),
        )

    def __call__(
            self,
            X: mlx.core.array,
            M: typing.Optional[mlx.core.array]=None,
            seq_len: typing.Optional[typing.Union[float, mlx.core.array]]=None,
    ) -> mlx.core.array:
        Y = self.layer_norm(X)
        Y = self.attention(Y, Y, Y, M=M, seq_len=seq_len)
        Y = self.dropout(Y)
        X = X + Y

        return X + self.fcffn(X)


class TransformerDecoderLayer(mlx.nn.Module):
    def __init__(
            self,
            h: int,
            d_m: int,
            d_ffn: typing.Optional[int]=None,
            dropout_p: float=0.1,
            attention_type: AttentionType=AttentionType.SOFTMAX,
            scaling: AttentionScaling=AttentionScaling.DIMS,
    ):
        super().__init__()

        d_ffn = d_ffn if d_ffn is not None else 4 * d_m

        self.layer_norm_1 = mlx.nn.LayerNorm(dims=d_m)
        self.self_attention = MultiHeadAttention(
            h=h,
            d_m=d_m,
            attention_type=attention_type,
            scaling=scaling,
        )
        self.dropout_1 = DropIn(max_dims=d_m, p=dropout_p)

        self.layer_norm_2 = mlx.nn.LayerNorm(dims=d_m)
        self.cross_attention = MultiHeadAttention(
            h=h,
            d_m=d_m,
            attention_type=attention_type,
            scaling=scaling,
        )
        self.dropout_2 = DropIn(max_dims=d_m, p=dropout_p)

        self.fcffn = mlx.nn.Sequential(
            mlx.nn.LayerNorm(dims=d_m),
            mlx.nn.Linear(input_dims=d_m, output_dims=d_ffn),
            mlx.nn.ReLU(),
            DropIn(max_dims=d_ffn, p=dropout_p),
            mlx.nn.Linear(input_dims=d_ffn, output_dims=d_m),
            DropIn(max_dims=d_m, p=dropout_p),
        )

    def __call__(
            self,
            X_Q: mlx.core.array,
            X_KV: mlx.core.array,
            M_Q: typing.Optional[mlx.core.array]=None,
            M_KV: typing.Optional[mlx.core.array]=None,
            seq_len_Q: typing.Optional[typing.Union[float, mlx.core.array]]=None,
            seq_len_KV: typing.Optional[typing.Union[float, mlx.core.array]]=None,
    ) -> mlx.core.array:
        Y = self.layer_norm_1(X_Q)
        Y = self.self_attention(Y, Y, Y, M=M_Q, seq_len=seq_len_Q)
        Y = self.dropout_1(Y)
        X_Q = X_Q + Y

        Y = self.layer_norm_2(X_Q)
        Y = self.cross_attention(
            X_Q=Y,
            X_K=X_KV,
            X_V=X_KV,
            M=M_KV,
            seq_len=seq_len_KV
        )
        Y = self.dropout_2(Y)
        X_Q = X_Q + Y

        return X_Q + self.fcffn(X_Q)


def test_transformer():
    b = 2
    d_m = 6
    d_q = 5
    d_kv = 7

    X_1 = mlx.core.random.uniform(shape=(b, d_q, d_m))
    X_2 = mlx.core.random.uniform(shape=(b, d_kv, d_m))
    M_1 = (
        make_causal_mask(shape=(d_q, d_q))
        + make_padding_mask(
            mlx.core.array([[False] * (d_q - 1) + [True]] * b)
        )
    )
    M_2 = make_padding_mask(
        mlx.core.array([[False] * (d_kv - 1) + [True]] * b)
    )

    for attention_type in AttentionType:
        X_Q, X_KV, M_Q, M_KV = X_1 * 1.0, X_2 * 1.0, M_1 * 1.0, M_2 * 1.0
        check_masking = True
        h = 3

        if attention_type == AttentionType.SIMPLIFIED_RELU:
            h = 1

        elif (
                attention_type == AttentionType.UNBIASED_PERCEPTRON
                or attention_type == AttentionType.PERCEPTRON
        ):
            h = 1
            X_Q, X_KV, M_Q, M_KV = X_1 * 1.0, X_1 * 1.0, M_1 * 1.0, M_1 * 1.0
            check_masking = False

        transformer = Transformer(
            n_encoders=3,
            n_decoders=3,
            h=h,
            d_m=d_m,
            attention_type=attention_type,
        )
        Y = transformer(X_Q=X_Q, X_KV=X_KV, M_Q=M_Q, M_KV=M_KV)
    
        assert Y.shape == (b, d_q, d_m), f"{attention_type=}, {Y.shape=}"
    
        mha = MultiHeadAttention(h=h, d_m=d_m, attention_type=attention_type)
        Y1 = mha(X_Q=X_Q, X_K=X_KV, X_V=X_KV, M=M_KV)
    
        assert Y1.shape == (b, d_q, d_m), f"{attention_type=}, {Y1.shape=}"

        if check_masking:
            X_KV[:, -1, :] = 9999.0
            Y2 = mha(X_Q=X_Q, X_K=X_KV, X_V=X_KV, M=M_KV)
        
            assert (Y1 != Y2).sum() < 0.1, f"{attention_type=}"


test_transformer()

#### Vision transformer

In [11]:
class VisionTransformer(mlx.nn.Module):
    def __init__(
            self,
            in_channels: int,
            patch_size: typing.Union[int, tuple[int, int]],
            num_layers: int,
            dims: int,
            num_heads: int,
            dims_ffn: int,
            dropout_p: float=0.1,
            attention_type: AttentionType=AttentionType.SOFTMAX,
            scaling: AttentionScaling=AttentionScaling.DIMS,
    ):
        super().__init__()

        self.result_token = mlx.core.random.normal((1, 1, dims)) * 0.02
        self.embedding = mlx.nn.Conv2d(
            in_channels=in_channels,
            out_channels=dims,
            kernel_size=patch_size,
            stride=patch_size,
        )
        self.positional_encoding = mlx.nn.SinusoidalPositionalEncoding(dims)
        self.dropout = DropIn(max_dims=dims, p=dropout_p)
        self.encoder = TransformerEncoder(
            n=num_layers,
            d_m=dims,
            h=num_heads,
            d_ffn=dims_ffn,
            dropout_p=dropout_p,
            attention_type=attention_type,
            scaling=scaling,
        )
    
    def __call__(self, X: mlx.core.array) -> mlx.core.array:
        # X shape: [b, h, w, c]

        batch_size = X.shape[0]

        result_tokens = mlx.core.repeat(
            self.result_token, batch_size, axis=0
        ) # Shape: [b, 1, d_m]

        
        X = self.embedding(X) # Shape: [b, h, w, d_m]

        # [b, h, w, d_m] --> [b, h * w, d_m]
        h, w, d_m = X.shape[1:]
        X = mlx.core.reshape(X, (batch_size, h * w, d_m))

        # Shape: [b, 1 + h * w, d_m]
        X = mlx.core.concatenate((result_tokens, X), axis=1)

        # Shape: [1, 1 + h * w, d_m]
        positions = self.positional_encoding(
            mlx.core.arange(1 + h * w)
        )[None, :, :]

        X = X + positions
        X = self.dropout(X)
        Y = self.encoder(X=X)

        return Y[:, 0, :]

#### Image processing

In [12]:
class ImageEditor:
    def __init__(self):
        self.brightness = None
        self.contrast = None
        self.old_temperature = None
        self.new_temperature = None
        self.sharpness = None
        self.transform_mtx = None
        self.transform_mtx_inv = None
        self.is_transformed = None
        self.noise_level = None

        self.clear()

    def clear(self):
        self.brightness = 1.0
        self.contrast = 1.0
        self.old_temperature = 6500.0
        self.new_temperature = 6500.0
        self.sharpness = 1.0
        self.transform_mtx = mlx.core.eye(3)
        self.transform_mtx_inv = mlx.core.eye(3)
        self.is_transformed = False
        self.noise_level = 0.0

    def set_brightness(self, brightness: float):
        self.brightness = float(brightness)

        return self

    def set_contrast(self, contrast: float):
        self.contrast = float(contrast)

        return self

    def change_temperature(self, old: float, new: float):
        self.old_temperature = float(old)
        self.new_temperature = float(new)

        return self

    def set_sharpness(self, sharpness: float):
        self.sharpness = sharpness

        return self

    def flip_horizontal(self):
        return self.transform(
            [
                [-1.0,  0.0,  0.0],
                [ 0.0,  1.0,  0.0],
                [ 0.0,  0.0,  1.0],
            ],
        )

    def transform(self, M):
        M = mlx.core.array(M, dtype=mlx.core.float32)
        self.transform_mtx = M @ self.transform_mtx
        self.is_transformed = True

        return self

    def flip_vertical(self):
        return self.transform(
            [
                [ 1.0,  0.0,  0.0],
                [ 0.0, -1.0,  0.0],
                [ 0.0,  0.0,  1.0],
            ],
        )

    def rotate(self, angle: float):
        if not self._is_significant(angle):
            return self

        theta = math.radians(float(angle))
        cos = math.cos(theta)
        sin = math.sin(theta)

        return self.transform(
            [
                [ cos, -sin,  0.0],
                [ sin,  cos,  0.0],
                [ 0.0,  0.0,  1.0],
            ],
        )

    def _is_significant(self, x: float) -> bool:
        return abs(x) > 1e-6

    def translate(self, dx: float, dy: float):
        dx = float(dx)
        dy = float(dy)

        if not (self._is_significant(dx) or self._is_significant(dy)):
            return self

        return self.transform(
            [
                [1.0, 0.0,  dx],
                [0.0, 1.0,  dy],
                [0.0, 0.0, 1.0],
            ],
        )

    def skew(self, x: float, y: float):
        x = float(x)
        y = float(y)

        if not (self._is_significant(x) or self._is_significant(y)):
            return self

        return self.transform(
            [
                [1.0,   x, 0.0],
                [  y, 1.0, 0.0],
                [0.0, 0.0, 1.0],
            ],
        )

    def scale(self, x: float, y: float):
        x = float(x)
        y = float(y)

        if not (self._is_significant(x - 1.0) or self._is_significant(y - 1.0)):
            return self

        return self.transform(
            [
                [  x, 0.0, 0.0],
                [0.0,   y, 0.0],
                [0.0, 0.0, 1.0],
            ],
        )

    def set_noise_level(self, noise_level: float):
        self.noise_level = float(noise_level)

        return self

    def edit(self, image: mlx.core.array, dither: float=0.005) -> mlx.core.array:
        if len(image.shape) < 3 or image.shape[-1] != 3:
            raise ValueError(
                f"Expected image(s) of shape [..., H, W, 3], got {tuple(image.shape)}"
            )

        image = image.astype(mlx.core.float32)
        image = self._apply_brightness(image)
        image = self._apply_contrast(image)
        image = self._apply_temperature(image)
        image = self._apply_sharpness(image)
        image = self._apply_transformation(image, dither)
        image = self._apply_noise(image)
        image = mlx.core.clip(image, 0.0, 1.0)

        return image

    def _apply_brightness(self, image: mlx.core.array) -> mlx.core.array:
        if self.brightness == 1.0:
            return image

        return self.brightness * image

    def _apply_contrast(self, image: mlx.core.array) -> mlx.core.array:
        luma = mlx.core.array([0.2126, 0.7152, 0.0722])
        gray = image @ luma
        mean = mlx.core.mean(gray, axis=(-2, -1), keepdims=True)[..., None]

        return mean + (image - mean) * self.contrast

    def _apply_temperature(self, image: mlx.core.array) -> mlx.core.array:
        if self.old_temperature == self.new_temperature:
            return image

        old_rgb = self._kelvin_to_rgb(self.old_temperature)
        new_rgb = self._kelvin_to_rgb(self.new_temperature)
        scale = mlx.core.array([n / o for o, n in zip(old_rgb, new_rgb)])

        return image * scale

    def _kelvin_to_rgb(self, k: float) -> tuple[float, float, float]:
        # https://tannerhelland.com/2012/09/18/convert-temperature-rgb-algorithm-code.html

        t = max(min(k, 40000.0), 1000.0) / 100.0
        red = 1.0
        green = 1.0
        blue = 1.0

        if t <= 66.0:
            red = 255.0
            green = 99.4708025861 * math.log(t) - 161.1195681661

            if t <= 19.0:
                blue = 0.0
            else:
                blue = t - 10
                blue = 138.5177312231 * math.log(blue) - 305.0447927307
        else:
            tm60 = t - 60.0
            red = 329.698727446 * (tm60 ** (-0.1332047592))
            green = 288.1221695283 * (tm60 ** (-0.0755148492))
            blue = 255.0

        return tuple(
            max(min(c, 255.0), 1.0) / 255.0
            for c in (red, green, blue)
        )

    def _apply_sharpness(self, image: mlx.core.array) -> mlx.core.array:
        if self.sharpness == 1.0:
            return image

        shape = image.shape
        channels = shape[-1]
        width = shape[-2]
        height = shape[-3]

        image = image.reshape(-1, height, width, channels)

        top = image[..., 0:1, :, :]
        bottom = image[..., -1:, :, :]
        image = mlx.core.concatenate(
            [top, top, image, bottom, bottom], axis=1
        )

        left = image[..., :, 0:1, :]
        right = image[..., :, -1:, :]
        image = mlx.core.concatenate(
            [left, left, image, right, right], axis=2
        )

        blur = mlx.core.ones((channels, 5, 5, 1))
        blur /= blur[0, :, :, 0].sum()
        blurred = mlx.core.conv2d(image, blur, groups=channels)
        image = image[..., 2:-2, 2:-2, :]
        image = blurred + self.sharpness * (image - blurred)

        return image.reshape(shape)

    def _apply_transformation(
            self,
            image: mlx.core.array,
            dither: float,
    ) -> mlx.core.array:
        if not self.is_transformed:
            self.transform_mtx_inv = mlx.core.eye(3)

            return image

        width = image.shape[-2]
        height = image.shape[-3]
        width_half = width * 0.5
        height_half = height * 0.5

        with mlx.core.stream(mlx.core.cpu):
            self.transform_mtx_inv = mlx.core.linalg.inv(self.transform_mtx)

        y, x = mlx.core.meshgrid(
            mlx.core.arange(height, dtype=mlx.core.float32) - height_half,
            mlx.core.arange(width, dtype=mlx.core.float32) - width_half,
            indexing="ij"
        )
        x = x.reshape(-1)
        y = y.reshape(-1)
        coords = mlx.core.stack([x, y, mlx.core.ones(x.shape)])

        coords = self.transform_mtx_inv @ coords
        x = (coords[0, :] / coords[2, :]).reshape(height, width) + width_half
        y = (coords[1, :] / coords[2, :]).reshape(height, width) + height_half

        return self._resample(image, x, y, width, height, dither)

    def _resample(
            self,
            image: mlx.core.array,
            x: mlx.core.array,
            y: mlx.core.array,
            width: int,
            height: int,
            dither: float,
    ) -> mlx.core.array:
        # Bilinear interpolation, see:
        # https://en.wikipedia.org/wiki/Bilinear_interpolation

        dither = float(dither)

        if self._is_significant(dither):
            image = image + dither * mlx.core.random.uniform(shape=image.shape)

        image = mlx.core.pad(
            image,
            [(0, 0)] * (len(image.shape) - 3) + [(1, 1), (1, 1), (0, 0)]
        )
        x += 1.0
        y += 1.0

        x_0 = mlx.core.floor(x)
        y_0 = mlx.core.floor(y)
        x_1 = x_0 + 1.0
        y_1 = y_0 + 1.0

        # A source pixel's weight is proportional to the diagonally
        # opposite rectangle. Last dimension is for the color channels.
        w_a = mlx.core.expand_dims((x_1 - x) * (y_1 - y), -1)
        w_b = mlx.core.expand_dims((x - x_0) * (y_1 - y), -1)
        w_c = mlx.core.expand_dims((x_1 - x) * (y - y_0), -1)
        w_d = mlx.core.expand_dims((x - x_0) * (y - y_0), -1)

        x_0 = mlx.core.clip(x_0, 0.0, width + 1.0).astype(mlx.core.int32)
        x_1 = mlx.core.clip(x_1, 0.0, width + 1.0).astype(mlx.core.int32)
        y_0 = mlx.core.clip(y_0, 0.0, height + 1.0).astype(mlx.core.int32)
        y_1 = mlx.core.clip(y_1, 0.0, height + 1.0).astype(mlx.core.int32)

        img_a = image[..., y_0, x_0, :]
        img_b = image[..., y_0, x_1, :]
        img_c = image[..., y_1, x_0, :]
        img_d = image[..., y_1, x_1, :]

        return w_a * img_a + w_b * img_b + w_c * img_c + w_d * img_d
    
    def _apply_noise(self, image: mlx.core.array) -> mlx.core.array:
        if not self._is_significant(self.noise_level):
            return image

        noise = 0.0
        scale = self.noise_level / 3.0

        for i in range(3):
            noise = noise + mlx.core.random.uniform(
                low=-scale, high=scale, shape=image.shape
            )

        return image + noise


def randomize_image(
        image: mlx.core.array,
        brightness: tuple[float, float, float]=(0.3, 0.8, 1.2),
        contrast: tuple[float, float, float]=(0.3, 0.5, 1.5),
        temperature: tuple[float, float, float]=(0.3, 3000.0, 10000.0),
        sharpness: tuple[float, float, float]=(0.3, 0.5, 1.5),
        flip_horizontal: float=0.3,
        flip_vertical: float=0.3,
        rotate: tuple[float, float, float]=(0.3, -20.0, 20.0),
        skew: tuple[float, float, float]=(0.2, -0.15, 0.15),
        scale: tuple[float, float, float]=(0.3, 0.8, 1.2),
        translate: tuple[float, float, float]=(0.3, -0.2, 0.2),
        noise: tuple[float, float, float]=(0.3, 0.05, 0.20),
) -> mlx.core.array:
    # Shape: [batch_size, height, width, channels] or [height, width, channels]

    editor = ImageEditor()

    rnd = lambda limits: (
        limits[0] + random.random() * (limits[1] - limits[0])
    )

    if random.random() < brightness[0]:
        editor.set_brightness(rnd(brightness[1:]))

    if random.random() < contrast[0]:
        editor.set_contrast(rnd(contrast[1:]))

    if random.random() < temperature[0]:
        editor.change_temperature(6500.0, rnd(temperature[1:]))

    if random.random() < sharpness[0]:
        editor.set_sharpness(rnd(sharpness[1:]))

    if random.random() < flip_horizontal:
        editor.flip_horizontal()

    if random.random() < flip_vertical:
        editor.flip_vertical()

    if random.random() < rotate[0]:
        editor.rotate(rnd(rotate[1:]))

    if random.random() < skew[0]:
        editor.skew(rnd(skew[1:]), rnd(skew[1:]))

    if random.random() < scale[0]:
        editor.scale(rnd(scale[1:]), rnd(scale[1:]))

    if random.random() < translate[0]:
        width = image.shape[-2]
        height = image.shape[-3]
    
        dx_min = translate[1] * width
        dx_max = translate[2] * width
        dy_min = translate[1] * height
        dy_max = translate[2] * height

        editor.translate(rnd((dx_min, dx_max)), rnd((dy_min, dy_max)))

    if random.random() < noise[0]:
        editor.set_noise_level(rnd(noise[1:]))

    image = editor.edit(image, dither=0.005)

    mlx.core.eval(image)

    return image

#### Text processing

In [13]:
TOKEN_RE = re.compile(r"\w+|<unk>|\n|[^\w\s]", flags=re.UNICODE | re.MULTILINE)
SPECIAL_TOKENS = ["<pad>", "<cls>", "<unk>", "<bos>", "<eos>", "\n"]

TOK_PAD = 0
TOK_CLS = 1
TOK_UNK = 2
TOK_BOS = 3
TOK_EOS = 4
TOK_BRK = 5


class TextEmbedding(mlx.nn.Module):
    def __init__(self, vocabulary_size: int, d_m: int):
        super().__init__()

        self.scale = math.sqrt(d_m)
        self.embedding = mlx.nn.Embedding(vocabulary_size, d_m)
        self.positional_encoding = mlx.nn.SinusoidalPositionalEncoding(d_m)

    def __call__(self, tokens: mlx.core.array) -> mlx.core.array:
        token_vectors = self.embedding(tokens) * self.scale
        positions = self.positional_encoding(
            mlx.core.arange(tokens.shape[-1])
        )[None, :, :]

        return token_vectors + positions


class Vocabulary:
    def __init__(
            self,
            texts: collections.abc.Iterable[str],
            min_occurrences: int=2,
            case_insensitive: bool=True,
    ):
        self.case_insensitive = case_insensitive
        tokens = self._find_all_tokens(
            texts,
            min_occurrences=min_occurrences,
        )
        self.tokens_by_id = (
            SPECIAL_TOKENS + [token for token, _ in tokens]
        )
        self.ids_by_token = {
            token: idx for idx, token in enumerate(self.tokens_by_id)
        }

    def _find_all_tokens(
            self,
            texts: collections.abc.Iterable[str],
            min_occurrences: int,
    ) -> collections.abc.Iterable[tuple[str, int]]:
        counter = collections.Counter()
        unk = SPECIAL_TOKENS[TOK_UNK]

        for text in texts:
            counter.update(
                (
                    token
                    for token in self._tokenize(text)
                    if token != unk
                )
            )

        # The main purpose of sorting is to ensure that the token ids
        # are assigned deterministically, though it would have been
        # useful to sort primarily by the number of occurrences.
        return sorted(
            (
                (token, occurrences)
                for token, occurrences in counter.items()
                if occurrences >= min_occurrences
            ),
            reverse=True,
        )

    def _tokenize(self, text: str) -> typing.List[int]:
        if self.case_insensitive:
            text = text.lower()

        text = text.replace("\r\n", "\n").replace("\r", "\n")

        return (token for token in TOKEN_RE.findall(text))
    
    def __len__(self) -> int:
        return len(self.tokens_by_id)

    def encode(self, text: str) -> typing.List[int]:
        return [
            self.ids_by_token.get(token, TOK_UNK)
            for token in self._tokenize(text)
        ]

    def decode(
            self,
            tokens: collections.abc.Iterable[int],
            stop_at_eos: bool=True
    ) -> typing.List[str]:
        text = []

        for token in tokens:
            if token < 0 or token >= len(self.tokens_by_id):
                token = TOK_UNK

            text.append(self.tokens_by_id[token])

            if token == TOK_EOS and stop_at_eos:
                break

        return text
            

def test_vocabulary():
    v_ci = Vocabulary(
        ["to be or", "not to", "be", "<unk> <unk> <unk>"],
        case_insensitive=True,
    )

    assert len(v_ci) == 8
    assert v_ci.tokens_by_id == SPECIAL_TOKENS + ["to", "be"]

    encoded = v_ci.encode("To be or not to <unk> <Unk> bE")
    decoded = " ".join(
        v_ci.decode([TOK_PAD, TOK_BOS] + encoded + [TOK_EOS, TOK_PAD])
    )
    assert decoded == "<pad> <bos> to be <unk> <unk> to <unk> <unk> be <eos>"


test_vocabulary()

### Snapshot management

In [14]:
def save_arrays(file_name: str, arrays: typing.Dict[str, mlx.core.array]):
    with gzip.open(file_name, "wb") as fp:
        mlx.core.save_safetensors(
            fp,
            mlx.utils.tree_flatten(arrays, destination={})
        )


def load_arrays(file_name: str) -> typing.Dict[str, mlx.core.array]:
    with gzip.open(file_name, "rb") as fp:
        return mlx.core.load(fp, format="safetensors")


class Replay:
    SAVE_TAG = "_save"
    TIME_TAG = "_time"

    def __init__(self, snapshot_file: typing.Optional[pathlib.Path]):
        self.events = []
        self.next_event_idx = 0
        self.snapshot_file = snapshot_file

        self._restore_state()

    def is_replaying(self) -> bool:
        return self.next_event_idx < len(self.events)

    def __call__(self, tag: str, func: typing.Callable) -> object:
        result, is_valid = self._fetch_next_event_result(tag)

        if not is_valid:
            result = func()
            self._store_event(tag, result)

        return result

    def time(self) -> float:
        result, is_valid = self._fetch_next_event_result(self.TIME_TAG)

        if not is_valid:
            result = self._now()
            self._store_event(self.TIME_TAG, result)

        return result

    def save(self):
        result, is_valid = self._fetch_next_event_result(self.SAVE_TAG)

        if is_valid or self.snapshot_file is None:
            return

        self._store_event(self.SAVE_TAG, self._now())
        now = self._now()
        events = [
            [event[0], event[1] - now]
            if event[0] == self.SAVE_TAG or event[0] == self.TIME_TAG
            else event
            for event in self.events
        ]

        with gzip.open(str(self.snapshot_file), "wb") as fp:
            fp.write(json.dumps(events).encode("utf-8"))

    def _restore_state(self):
        if self.snapshot_file is None or not self.snapshot_file.exists():
            return

        with gzip.open(str(self.snapshot_file), "rb") as fp:
            serialized = fp.read().decode("utf-8")

        if serialized == "":
            return

        events = json.loads(serialized)

        now = self._now()
        self.events = [
            [event[0], event[1] + now]
            if event[0] == self.SAVE_TAG or event[0] == self.TIME_TAG
            else event
            for event in events
        ]
        self.next_event_idx = 0

    def _fetch_next_event_result(self, expected_tag: str) -> tuple[object, bool]:
        if not self.is_replaying():
            return None, False

        actual_tag, result = self.events[self.next_event_idx]

        if expected_tag == actual_tag:
            self.next_event_idx += 1

            return result, True

        self.events = self.events[:self.next_event_idx]

        return None, False

    def _store_event(self, tag: str, result):
        if self.snapshot_file is not None:
            self.events.append([tag, result])
            self.next_event_idx = len(self.events)

    def _now(self) -> float:
        return time.time()


def test_replay():
    class TestableReplay(Replay):
        def __init__(self, current_time: float, *args, **kwargs):
            self.current_time = current_time

            super().__init__(*args, **kwargs)

        def _now(self) -> float:
            return self.current_time

    snapshot_file_fd, snapshot_file_name = tempfile.mkstemp(
        prefix="replay-test-",
    )
    os.close(snapshot_file_fd)
    snapshot_file = pathlib.Path(snapshot_file_name)

    try:
        expensive_results = [f"result-{i + 1}" for i in range(100)]
        expensive_fn = lambda: expensive_results.pop(0)

        replay_1 = TestableReplay(1.0, snapshot_file)
        is_replaying_1 = replay_1.is_replaying()
        result_1 = replay_1("first", expensive_fn)
        replay_1.current_time = 2.0
        time_1 = replay_1.time()
        result_2 = replay_1("second", expensive_fn)
        result_3 = replay_1("third", expensive_fn)
        is_replaying_2 = replay_1.is_replaying()
        replay_1.current_time = 7.0
        time_2 = replay_1.time()
        replay_1.current_time = 9.0
        replay_1.save()
        result_4 = replay_1("fourth", expensive_fn)
        result_5 = replay_1("fifth", expensive_fn)
        replay_1.current_time = 10.0
        time_3 = replay_1.time()

        assert result_1 == "result-1", f"Expected result-1, got {result_1!r}"
        assert result_2 == "result-2", f"Expected result-2, got {result_2!r}"
        assert result_3 == "result-3", f"Expected result-3, got {result_3!r}"
        assert result_4 == "result-4", f"Expected result-4, got {result_4!r}"
        assert result_5 == "result-5", f"Expected result-5, got {result_5!r}"

        assert time_1 == 2.0, f"Expected 2.0, got {time_1!r}"
        assert time_2 == 7.0, f"Expected 7.0, got {time_2!r}"
        assert time_3 == 10.0, f"Expected 10.0, got {time_3!r}"

        assert is_replaying_1 == False
        assert is_replaying_2 == False

        replay_2 = TestableReplay(100.0, snapshot_file)
        replay_1.save() # too late for replay_2 to see result 4 and 5
        is_replaying_3 = replay_2.is_replaying()
        result_1_replayed_1 = replay_2("first", expensive_fn)
        time_1_replayed_1 = replay_2.time()
        result_2_replayed_1 = replay_2("second", expensive_fn)
        result_3_replayed_1 = replay_2("third", expensive_fn)
        replay_2.current_time = 102.0
        time_2_replayed_1 = replay_2.time()
        replay_2.current_time = 103.0
        is_replaying_4 = replay_2.is_replaying()
        replay_2.save() # no-op, still catching up
        is_replaying_5 = replay_2.is_replaying()
        result_6 = replay_2("fourth", expensive_fn) # not saved in time
        result_7 = replay_2("fifth", expensive_fn)  # not saved in time
        replay_2.current_time = 105.0
        time_4 = replay_2.time()

        assert result_1_replayed_1 == "result-1", f"Expected result-1, got {result_1_replayed_1!r}"
        assert result_2_replayed_1 == "result-2", f"Expected result-2, got {result_2_replayed_1!r}"
        assert result_3_replayed_1 == "result-3", f"Expected result-3, got {result_3_replayed_1!r}"
        assert result_6 == "result-6", f"Expected result-6, got {result_6!r}"
        assert result_7 == "result-7", f"Expected result-7, got {result_7!r}"

        assert time_1_replayed_1 == 93.0, f"Expected 93.0, got {time_1_replayed_1!r}"
        assert time_2_replayed_1 == 98.0, f"Expected 98.0, got {time_2_replayed_1!r}"
        assert time_4 == 105.0, f"Expected 105.0, got {time_4!r}"

        assert is_replaying_3 == True
        assert is_replaying_4 == True
        assert is_replaying_5 == False

        replay_3 = TestableReplay(200.0, snapshot_file)
        result_1_replayed_2 = replay_3("first", expensive_fn)
        replay_3.time()
        result_2_replayed_2 = replay_3("second", expensive_fn)
        is_replaying_6 = replay_3.is_replaying()
        result_8 = replay_3("what", expensive_fn) # replay_3 diverged
        is_replaying_7 = replay_3.is_replaying()

        assert result_1_replayed_2 == "result-1", f"Expected result-1, got {result_1_replayed_2!r}"
        assert result_2_replayed_2 == "result-2", f"Expected result-2, got {result_2_replayed_2!r}"
        assert result_8 == "result-8", f"Expected result-8, got {result_8!r}"

        assert is_replaying_6 == True
        assert is_replaying_7 == False

        replay_4 = TestableReplay(300.0, snapshot_file)
        replay_4("first", expensive_fn)
        time_1_replayed_2 = replay_4.time()
        replay_4("second", expensive_fn)
        replay_4("third", expensive_fn)
        time_2_replayed_2 = replay_4.time()
        replay_4.save()
        result_4_replayed_1 = replay_4("fourth", expensive_fn)
        result_5_replayed_1 = replay_4("fifth", expensive_fn)
        time_3_replayed_1 = replay_4.time()

        assert result_4_replayed_1 == "result-4", f"Expected result-4, got {result_4_replayed_1!r}"
        assert result_5_replayed_1 == "result-5", f"Expected result-5, got {result_5_replayed_1!r}"

        assert time_1_replayed_2 == 292.0, f"Expected 292.0, got {time_1_replayed_2!r}"
        assert time_2_replayed_2 == 297.0, f"Expected 297.0, got {time_2_replayed_2!r}"
        assert time_3_replayed_1 == 300.0, f"Expected 300.0, got {time_3_replayed_1!r}"

        replay_2.save() # replay_2 diverged, state gets overwritten
        replay_5 = TestableReplay(400.0, snapshot_file)
        replay_5("first", expensive_fn)
        replay_5.time()
        replay_5("second", expensive_fn)
        replay_5("third", expensive_fn)
        replay_5.time()
        replay_5.save()
        result_6_replayed_1 = replay_5("fourth", expensive_fn)
        result_7_replayed_1 = replay_5("fifth", expensive_fn)

        assert result_6_replayed_1 == "result-6", f"Expected result-6, got {result_6_replayed_1!r}"
        assert result_7_replayed_1 == "result-7", f"Expected result-7, got {result_7_replayed_1!r}"

        replay_none_1 = TestableReplay(500.0, None)
        result_9 = replay_none_1("first", expensive_fn)
        time_5 = replay_none_1.time()
        replay_none_1.save()
        replay_none_2 = TestableReplay(600.0, None)
        result_10 = replay_none_2("first", expensive_fn)
        time_6 = replay_none_2.time()
        is_replaying_8 = replay_none_2.is_replaying()

        assert result_9 == "result-9", f"Expected result-9, got {result_9!r}"
        assert result_10 == "result-10", f"Expected result-10, got {result_10!r}"

        assert time_5 == 500.0, f"Expected 500.0, got {time_5!r}"
        assert time_6 == 600.0, f"Expected 600.0, got {time_6!r}"

        assert is_replaying_8 == False

    finally:
        snapshot_file.unlink()


test_replay()

### Training loop

In [15]:
def run_training(
        model: mlx.nn.Module,
        loss_fn: typing.Callable,
        train_batch_fn: typing.Callable[[int], typing.Iterable[tuple]],
        test_batch_fn: typing.Callable[[int], typing.Iterable[tuple]],
        num_epochs: int,
        num_grad_acc_batches: int=1,
        lr: float=0.001,
        weight_decay: float=0.01,
        num_status_batches: int=10,
        status_batches_avg_window_len: int=0,
        snapshot_dir: typing.Optional[pathlib.Path]=None,
        num_snapshot_batches: int=0,
        train_aug_fn: typing.Optional[typing.Callable[[tuple], tuple]]=None,
        test_aug_fn: typing.Optional[typing.Callable[[tuple], tuple]]=None,
        log_path: typing.Optional[pathlib.Path]=None,
        opt: typing.Optional[mlx.optimizers.Optimizer]=None,
) -> tuple[typing.List[tuple[float, float]], typing.List[tuple[float, float]]]:
    status_batches_avg_window_len = (
        status_batches_avg_window_len
        if status_batches_avg_window_len > 0
        else num_status_batches
    )

    if opt is None:
        opt = mlx.optimizers.AdamW(
            learning_rate=lr,
            weight_decay=weight_decay,
        )

    loss_and_grad_fn = mlx.nn.value_and_grad(model, loss_fn)
    model_file_name = None
    opt_file_name = None
    epoch_losses = []
    epoch_times = []
    replay = None

    if snapshot_dir is None:
        replay = Replay(None)
        num_snapshot_batches = 0
    else:
        replay = Replay(snapshot_dir / "train.json.gz")
        model_file_name = str(snapshot_dir / "model.safetensors.gz")
        opt_file_name = str(snapshot_dir / "opt.safetensors.gz")
        snapshot_dir.mkdir(parents=True, exist_ok=True)

        if (
                os.path.exists(model_file_name)
                and os.path.exists(opt_file_name)
        ):
            print(
                f"  Loading {model_file_name!r} and {opt_file_name}..."
            )

            model.load_weights(
                load_arrays(model_file_name).items(),
                strict=True,
            )
            mlx.core.eval(model.state)

            opt.state = mlx.utils.tree_unflatten(
                load_arrays(opt_file_name)
            )
            mlx.core.eval(opt.state)

    for epoch in range(num_epochs):
        epoch_train_loss, epoch_train_time = train_epoch(
            num_epochs=num_epochs,
            epoch=epoch,
            model=model,
            opt=opt,
            loss_and_grad_fn=loss_and_grad_fn,
            train_batch_fn=train_batch_fn,
            num_grad_acc_batches=num_grad_acc_batches,
            num_status_batches=num_status_batches,
            status_batches_avg_window_len=status_batches_avg_window_len,
            model_file_name=model_file_name,
            opt_file_name=opt_file_name,
            num_snapshot_batches=num_snapshot_batches,
            replay=replay,
            train_aug_fn=train_aug_fn,
            log_path=log_path,
        )        
        epoch_test_loss, epoch_test_time = test_epoch(
            num_epochs=num_epochs,
            epoch=epoch,
            model=model,
            loss_fn=loss_fn,
            test_batch_fn=test_batch_fn,
            num_status_batches=num_status_batches,
            status_batches_avg_window_len=status_batches_avg_window_len,
            num_snapshot_batches=num_snapshot_batches,
            replay=replay,
            test_aug_fn=test_aug_fn,
            log_path=log_path,
        )

        epoch_info = f"{epoch + 1}/{num_epochs}"
        train_time_info = f"{epoch_train_time:.1f}s"
        test_time_info = f"{epoch_test_time:.1f}s"
        log_msg(
            f"  Epoch  E:{epoch_info:<7}"
            f"  Train/T={train_time_info:<9}"
            f"  Train/E|L|={epoch_train_loss:.5f}"
            f"  Test/T={test_time_info:<9}"
            f"  Test/E|L|={epoch_test_loss:.5f}",
            log_path=log_path,
        )

        epoch_losses.append((epoch_train_loss, epoch_test_loss))
        epoch_times.append((epoch_train_time, epoch_test_time))

    print("")

    return epoch_losses, epoch_times


def train_epoch(
        num_epochs: int,
        epoch: int,
        model: mlx.nn.Module,
        opt: mlx.optimizers.Optimizer,
        loss_and_grad_fn: typing.Callable,
        train_batch_fn: typing.Callable[[int], typing.Iterable[tuple]],
        num_grad_acc_batches: int,
        num_status_batches: int,
        status_batches_avg_window_len: int,
        model_file_name: typing.Optional[str],
        opt_file_name: typing.Optional[str],
        num_snapshot_batches: int,
        replay: Replay,
        train_aug_fn: typing.Optional[typing.Callable[[tuple], tuple]],
        log_path: typing.Optional[pathlib.Path],
) -> tuple[float, float]:
    num_batches = 0
    grad_acc = None
    num_grad_acc = 0

    def train_batch(X: mlx.core.array, Y: mlx.core.array):
        nonlocal grad_acc, num_grad_acc

        batch_loss, grad = loss_and_grad_fn(model, X, Y)
        num_grad_acc += 1
        grad_acc = (
            grad
            if grad_acc is None
            else mlx.utils.tree_map(
                lambda a, g: a + g,
                grad_acc,
                grad,
            )
        )
        mlx.core.eval(grad_acc)

        if num_batches % num_grad_acc_batches == 0:
            grad_acc = mlx.utils.tree_map(lambda a: a / num_grad_acc, grad_acc)
            opt.update(model, grad_acc)
            mlx.core.eval(model.parameters())
            mlx.core.eval(opt.state)
            grad_acc = None
            num_grad_acc = 0

        return float(batch_loss.item()), grad_acc is None

    def save_snapshot():
        print(
            f"    Saving {model_file_name!r} and {opt_file_name!r}..."
        )

        save_arrays(model_file_name, model.parameters())
        save_arrays(opt_file_name, opt.state)

    model.train()

    epoch_info = f"{epoch + 1}/{num_epochs}"
    epoch_train_loss = 0.0
    batch_losses = []
    need_snapshot = False

    epoch_train_start = replay.time()
    print(
        f"    Start  E:{epoch_info:<7}"
        f"  {timestamp_to_str(epoch_train_start)}"
    )

    for X, Y in train_batch_fn(epoch):
        num_batches += 1

        if train_aug_fn is not None and not replay.is_replaying():
            X, Y = train_aug_fn(X, Y)

        batch_loss, grad_acc_is_clear = replay(
            "trnbl", lambda: train_batch(X, Y)
        )
        epoch_train_loss += batch_loss

        need_snapshot = (
            need_snapshot
            or (
                model_file_name is not None
                and opt_file_name is not None
                and num_snapshot_batches >= 1
                and num_batches % num_snapshot_batches == 0
            )
        )

        if need_snapshot and grad_acc_is_clear:
            need_snapshot = False
            replay.save()

            if not replay.is_replaying():
                save_snapshot()

        if num_status_batches < 1:
            continue

        batch_losses.append(batch_loss)

        if num_batches % num_status_batches != 0:
            continue

        batch_losses = batch_losses[-status_batches_avg_window_len:]
        batch_losses_avg = sum(batch_losses) / len(batch_losses)
        log_msg(
            f"    Train  E:{epoch_info:<7}"
            f"  B:{num_batches:<7}"
            f"  E|L|={batch_losses_avg:.5f}",
            log_path=log_path,
        )

    if grad_acc is not None and num_grad_acc > 0:
        grad_acc = mlx.utils.tree_map(lambda a: a / num_grad_acc, grad_acc)
        opt.update(model, grad_acc)
        mlx.core.eval(model.parameters())
        mlx.core.eval(opt.state)
        grad_acc = None

    epoch_train_end = replay.time()
    print(
        f"    End    E:{epoch_info:<7}"
        f"  {timestamp_to_str(epoch_train_end)}"
    )

    replay.save()

    if (
            model_file_name is not None
            and opt_file_name is not None
            and not replay.is_replaying()
    ):
        save_snapshot()

    epoch_train_loss /= max(num_batches, 1)
    epoch_train_time = epoch_train_end - epoch_train_start

    return epoch_train_loss, epoch_train_time


def test_epoch(
        num_epochs: int,
        epoch: int,
        model: mlx.nn.Module,
        loss_fn: typing.Callable,
        test_batch_fn: typing.Callable[[int], typing.Iterable[tuple]],
        num_status_batches: int,
        status_batches_avg_window_len: int,
        num_snapshot_batches: int,
        replay: Replay,
        test_aug_fn: typing.Optional[typing.Callable[[tuple], tuple]],
        log_path: typing.Optional[pathlib.Path],
) -> tuple[float, float]:
    model.eval()

    epoch_info = f"{epoch + 1}/{num_epochs}"
    epoch_test_loss = 0.0
    num_batches = 0
    batch_losses = []

    epoch_test_start = replay.time()
    print(
        f"    Start  E:{epoch_info:<7}"
        f"  {timestamp_to_str(epoch_test_start)}"
    )

    for X, Y in test_batch_fn(epoch):
        num_batches += 1

        if test_aug_fn is not None and not replay.is_replaying():
            X, Y = test_aug_fn(X, Y)

        batch_loss = replay(
            "tstbl", lambda: float(loss_fn(model, X, Y).item())
        )
        epoch_test_loss += batch_loss

        if (
                num_snapshot_batches >= 1
                and num_batches % num_snapshot_batches == 0
        ):
            replay.save()

        if num_status_batches < 1:
            continue
            
        batch_losses.append(batch_loss)

        if num_batches % num_status_batches != 0:
            continue

        batch_losses = batch_losses[-status_batches_avg_window_len:]
        batch_losses_avg = sum(batch_losses) / len(batch_losses)
        log_msg(
            f"    Test   E:{epoch_info:<7}"
            f"  B:{num_batches}"
            f"  E|L|={batch_losses_avg:.5f}",
            log_path=log_path,
        )

    epoch_test_end = replay.time()
    print(
        f"    End    E:{epoch_info:<7}"
        f"  {timestamp_to_str(epoch_test_end)}"
    )

    replay.save()

    epoch_test_loss /= max(num_batches, 1)
    epoch_test_time = epoch_test_end - epoch_test_start

    return epoch_test_loss, epoch_test_time


def timestamp_to_str(timestamp: float) -> str:
    return datetime.datetime.fromtimestamp(int(timestamp)).isoformat()

### Batches

In [16]:
def collate(
        batch_samples: typing.List[tuple],
) -> tuple[mlx.core.array, mlx.core.array]:
    X = mlx.core.array([x for x, _ in batch_samples])
    Y = mlx.core.array([y for _, y in batch_samples])

    return X, Y


def generate_batches(
        epoch: int,
        samples: typing.List[tuple],
        batch_size: int,
        shuffle=False,
        collate_fn: typing.Callable[
            [typing.List[tuple]],
            tuple[object, mlx.core.array],
        ]=collate,
):
    indices = list(range(len(samples)))

    if shuffle:
        random.shuffle(indices)

    for i in range(0, len(indices), batch_size):
        start = i
        end = i + batch_size

        yield collate_fn([samples[i] for i in indices[start:end]])


def generate_batches_drop_in(
        model: mlx.nn.Module,
        epoch: int,
        train_batch_fn: typing.Callable,
        drop_in_schedule: typing.Sequence[tuple[int, int, typing.Union[int, float]]],
        log_path: typing.Optional[pathlib.Path]=None,
):
    assert drop_in_schedule[0][0] == 0
    assert drop_in_schedule[0][1] == 0

    next_drop_in_idx = set_drop_in(
        model=model,
        drop_in_schedule=drop_in_schedule,
        next_drop_in_idx=0,
        epoch=epoch,
        batch=0,
        log_path=log_path,
    )

    for batch, (X, Y) in enumerate(train_batch_fn(epoch)):
        next_drop_in_idx = set_drop_in(
            model=model,
            drop_in_schedule=drop_in_schedule,
            next_drop_in_idx=next_drop_in_idx,
            epoch=epoch,
            batch=batch,
            log_path=log_path,
        )

        yield X, Y


def set_drop_in(
        model: mlx.nn.Module,
        drop_in_schedule: typing.Sequence[tuple[int, int, typing.Union[int, float]]],
        next_drop_in_idx: int,
        epoch: int,
        batch: int,
        log_path: typing.Optional[pathlib.Path]=None,
):
    changed = False

    while (
            next_drop_in_idx < len(drop_in_schedule)
            and (
                drop_in_schedule[next_drop_in_idx][0] < epoch
                or (
                    drop_in_schedule[next_drop_in_idx][0] == epoch
                    and drop_in_schedule[next_drop_in_idx][1] <= batch
                )
            )
    ):
        drop_in = drop_in_schedule[next_drop_in_idx][2]
        next_drop_in_idx += 1
        changed = True

    if changed:
        log_msg(f"    Drop-in: {drop_in:.3f}", log_path=log_path)
        DropIn.set_dims(model, drop_in)

    return next_drop_in_idx

### Loss functions

In [17]:
def cross_entropy_loss(
        model: mlx.nn.Module,
        X: mlx.core.array,
        Y_true: mlx.core.array,
) -> mlx.core.array:
    logits = model(X)

    return mlx.nn.losses.cross_entropy(logits, Y_true, reduction="mean")


def binary_cross_entropy_loss(
        model: mlx.nn.Module,
        X: mlx.core.array,
        Y_true: mlx.core.array,
) -> mlx.core.array:
    logits = model(X)

    return mlx.nn.losses.binary_cross_entropy(logits, Y_true, reduction="mean")

### Models

#### Causal language model

In [18]:
class CausalLanguageModel(mlx.nn.Module):
    def __init__(
            self,
            vocabulary_size: int,
            num_layers: int,
            dims: int,
            num_heads: int,
            dims_ffn: int,
            dropout_p: float=0.1,
            attention_type: AttentionType=AttentionType.SOFTMAX,
            scaling: AttentionScaling=AttentionScaling.DIMS,
    ):
        super().__init__()

        self.embedding = TextEmbedding(vocabulary_size, dims)
        self.dropout = DropIn(max_dims=dims, p=dropout_p)
        self.encoder = TransformerEncoder(
            n=num_layers,
            d_m=dims,
            h=num_heads,
            d_ffn=dims_ffn,
            dropout_p=dropout_p,
            attention_type=attention_type,
            scaling=scaling,
        )
        self.output = mlx.nn.Linear(dims, vocabulary_size)

    def __call__(self, tokens: mlx.core.array) -> mlx.core.array:
        Y = self._encode(tokens)

        return self.output(Y)

    def _encode(self, tokens: mlx.core.array) -> mlx.core.array:
        seq_len = mlx.core.arange(tokens.shape[-1])[None, :] + 1
        M = make_causal_mask((tokens.shape[-1], tokens.shape[-1]))
        X = self.embedding(tokens)
        X = self.dropout(X)
        Y = self.encoder(X=X, M=M, seq_len=seq_len)

        return Y

    def predict_next_token(self, tokens: mlx.core.array) -> mlx.core.array:
        Y = self._encode(tokens)

        return self.output(Y[..., -1, :])


def generate_causal_lm_batches(
        epoch: int,
        texts: typing.List[str],
        seq_len: int,
        vocabulary: Vocabulary,
        batch_size: int,
        shuffle=False,
):
    text_indices = list(range(len(texts)))

    if shuffle:
        random.shuffle(text_indices)

    chunks = []

    for i in text_indices:
        text = texts[i]
        encoded = [TOK_BOS] + vocabulary.encode(text) + [TOK_EOS]

        for j in range(0, len(encoded), seq_len):
            start = j
            end = j + seq_len + 1
            chunk = encoded[start:end]
            chunk += [TOK_PAD] * (seq_len + 1 - len(chunk))
            chunks.append(chunk)

        if len(chunks) > batch_size * 100:
            if shuffle:
                random.shuffle(chunks)

            while len(chunks) > batch_size:
                batch, chunks = chunks[0:batch_size], chunks[batch_size:]
                batch = mlx.core.array(batch)

                yield batch[:, 0:-1], batch[:, 1:]

    if shuffle:
        random.shuffle(chunks)

    for i in range(0, len(chunks), batch_size):
        start = i
        end = i + batch_size
        batch = chunks[start:end]
        batch = mlx.core.array(batch)

        yield batch[:, 0:-1], batch[:, 1:]


def valid_cross_entropy_loss(
        model: mlx.nn.Module,
        X: mlx.core.array,
        Y_true: mlx.core.array,
) -> mlx.core.array:
    logits = model(X)
    is_valid = Y_true != TOK_PAD
    losses = mlx.nn.losses.cross_entropy(logits, Y_true, reduction="none")

    return mlx.core.sum(losses * is_valid) / mlx.core.sum(is_valid)


def generate_text(
        model: mlx.nn.Module,
        vocabulary: Vocabulary,
        prompt: str,
        max_length: int,
        temperature: float=1.0,
) -> str:
    X = mlx.core.array([TOK_BOS] + vocabulary.encode(prompt))

    model.eval()
    temperature = max(temperature, 1e-3)
    generated_tokens = []

    while len(X) < max_length:
        Y = model.predict_next_token(X[None, ...]) / temperature
        next_token = mlx.core.random.categorical(Y[0])

        if next_token == TOK_EOS:
            break

        generated_tokens.append(next_token.item())
        X = mlx.core.concatenate([X, next_token])
        mlx.core.eval(X)

    return " ".join(vocabulary.decode(generated_tokens))


def test_generate_causal_lm_batches():
    texts = [
        "The quick, brown fox jumps over the lazy dog.",
        "Lorem ipsum dolor sit amet, consectetur adipisicing",
    ]
    vocabulary = Vocabulary(texts, min_occurrences=1)
    batches = generate_causal_lm_batches(
        0,
        texts,
        seq_len=5,
        vocabulary=vocabulary,
        batch_size=3,
    )
    decoded_batches = []

    for X, Y in batches:
        batch = []

        for i in range(X.shape[0]):
            batch.append(
                (
                    " ".join(vocabulary.decode(X[i].tolist(), stop_at_eos=False)),
                    " ".join(vocabulary.decode(Y[i].tolist(), stop_at_eos=False))
                )
            )

        decoded_batches.append(batch)

    expected = [
        [
            ("<bos> the quick , brown", "the quick , brown fox"),
            ("fox jumps over the lazy", "jumps over the lazy dog"),
            ("dog . <eos> <pad> <pad>", ". <eos> <pad> <pad> <pad>"),
        ],
        [
            ("<bos> lorem ipsum dolor sit", "lorem ipsum dolor sit amet"),
            ("amet , consectetur adipisicing <eos>", ", consectetur adipisicing <eos> <pad>"),
        ],
    ]
    assert decoded_batches == expected


test_generate_causal_lm_batches()

#### Image classifier

In [19]:
def collate_image_classification_samples(
        batch_samples: typing.List[tuple],
        num_classes: int,
) -> tuple[mlx.core.array, mlx.core.array]:
    return collate(
        [
            (
                x,
                [1.0 if y == i else 0.0 for i in range(num_classes)]
                if num_classes > 1
                else [y]
            )
            for x, y in batch_samples
        ]
    )


class ImageClassifier(mlx.nn.Module):
    def __init__(
            self,
            in_channels: int,
            patch_size: typing.Union[int, tuple[int, int]],
            num_layers: int,
            dims: int,
            num_heads: int,
            dims_ffn: int,
            num_classes: int,
            dropout_p: float=0.1,
            attention_type: AttentionType=AttentionType.SOFTMAX,
            scaling: AttentionScaling=AttentionScaling.DIMS,
    ):
        super().__init__()

        self.vision_transformer = VisionTransformer(
            in_channels=in_channels,
            patch_size=patch_size,
            num_layers=num_layers,
            dims=dims,
            num_heads=num_heads,
            dims_ffn=dims_ffn,
            dropout_p=dropout_p,
            attention_type=attention_type,
            scaling=scaling,
        )
        self.classify = mlx.nn.Linear(dims, num_classes)

    def __call__(self, X: mlx.core.array) -> mlx.core.array:
        Y = self.vision_transformer(X)
        Y = self.classify(Y)

        return Y

#### Text classifier

In [20]:
class TextClassificationMethod(str, enum.Enum):
    MEAN_POOLING = "mean-pooling"
    CLS_TOKEN = "cls-token"


def collate_text_classification_samples(
        vocabulary: Vocabulary,
        batch_samples: typing.List[tuple],
        num_classes: int,
) -> tuple[mlx.core.array, mlx.core.array]:
    encoded_batch_samples = [
        (
            vocabulary.encode(x),
            [1.0 if y == i else 0.0 for i in range(num_classes)]
            if num_classes > 1
            else [y]
        )
        for x, y in batch_samples
    ]
    max_len = max(len(X) for X, _ in encoded_batch_samples)
    padded_batch_samples = [
        (
            mlx.core.array(X + [TOK_PAD] * (max_len - len(X))),
            mlx.core.array(Y)
        )
        for X, Y in encoded_batch_samples
    ]

    return collate(padded_batch_samples)


class TextClassifier(mlx.nn.Module):
    def __init__(
            self,
            vocabulary_size: int,
            num_layers: int,
            dims: int,
            num_heads: int,
            dims_ffn: int,
            num_classes: int,
            dropout_p: float=0.1,
            attention_type: AttentionType=AttentionType.SOFTMAX,
            scaling: AttentionScaling=AttentionScaling.DIMS,
            mask_padding: bool=True,
            classification_method: TextClassificationMethod=TextClassificationMethod.MEAN_POOLING,
    ):
        super().__init__()

        if (
                classification_method != TextClassificationMethod.MEAN_POOLING
                and classification_method != TextClassificationMethod.CLS_TOKEN
        ):
            raise ValueError(
                "classification_method must be one of "
                + ", ".join(str(cm) for cm in TextClassificationMethod)
                + "; got {classification_method!r}"
            )

        self.embedding = TextEmbedding(vocabulary_size, dims)
        self.dropout = DropIn(max_dims=dims, p=dropout_p)
        self.encoder = TransformerEncoder(
            n=num_layers,
            d_m=dims,
            h=num_heads,
            d_ffn=dims_ffn,
            dropout_p=dropout_p,
            attention_type=attention_type,
            scaling=scaling,
        )
        self.classify = mlx.nn.Linear(dims, num_classes)
        self.mask_padding = mask_padding
        self.use_cls_token = (
            classification_method == TextClassificationMethod.CLS_TOKEN
        )

    def __call__(self, tokens: mlx.core.array) -> mlx.core.array:
        if self.use_cls_token:
            tokens_with_cls = mlx.core.zeros(
                (tokens.shape[0], tokens.shape[1] + 1),
                dtype=mlx.core.int32,
            )
            tokens_with_cls[:, 0] = TOK_CLS
            tokens_with_cls[:, 1:] = tokens
            tokens = tokens_with_cls

        is_padding = tokens == TOK_PAD
        is_valid = ~is_padding
        seq_len = is_valid.sum(axis=-1)
        M = make_padding_mask(is_padding) if self.mask_padding else None
        X = self.embedding(tokens)
        X = self.dropout(X)
        Y = self.encoder(X=X, M=M, seq_len=seq_len)

        if self.use_cls_token:
            return self.classify(Y[:, 0])

        is_valid = is_valid.astype(X.dtype)[..., None]
        mean_pooled = (
            mlx.core.sum(Y * is_valid, axis=1)
            / mlx.core.maximum(mlx.core.sum(is_valid, axis=1), 1.0)
        )

        return self.classify(mean_pooled)

### Metrics

In [21]:
def eval_classifier(
        model: mlx.nn.Module,
        test_batch_fn: typing.Callable[[int], typing.Iterable[tuple]],
        size_key_filters: typing.Optional[typing.Dict[str, bool]]=None,
) -> tuple[int, float, float, typing.List, float]:
    model.eval()

    all_cls_true = []
    all_cls_pred = []
    batch_lengths = []
    batch_accuracies = []
    num_classes = 2

    for X, Y_true in test_batch_fn(0):
        X_len = X.shape[1]
        Y_pred = model(X)

        if Y_true.shape[1] <= 1:
            cls_true = Y_true.reshape(-1)
            cls_pred = (
                (mlx.core.sigmoid(Y_pred) >= 0.5)
                    .astype(mlx.core.int32)
                    .reshape(-1)
            )
        else:
            num_classes = Y_true.shape[1]
            cls_true = mlx.core.argmax(Y_true, axis=1)
            cls_pred = mlx.core.argmax(Y_pred, axis=1)

        all_cls_true.append(cls_true)
        all_cls_pred.append(cls_pred)

        batch_lengths.append(X.shape[1])
        batch_accuracies.append(
            sklearn.metrics.accuracy_score(
                np.array(cls_true, copy=False),
                np.array(cls_pred, copy=False),
            )
        )

    size = compute_model_size(model, size_key_filters)

    Y_true = np.concatenate(all_cls_true)
    Y_pred = np.concatenate(all_cls_pred)
    accuracy = sklearn.metrics.accuracy_score(Y_true, Y_pred)
    f1 = sklearn.metrics.f1_score(
        Y_true, Y_pred, average="macro", zero_division=0.0
    )
    conf_mtx = sklearn.metrics.confusion_matrix(
        Y_true, Y_pred, labels=list(range(num_classes))
    )

    len_acc_corr = np.nan

    if len(batch_lengths) > 0:
        batch_lengths = np.array(batch_lengths)
        batch_accuracies = np.array(batch_accuracies)

        if batch_lengths.std() > 1e-6 and batch_accuracies.std() > 1e-6:
            len_acc_corr = float(
                np.corrcoef(batch_lengths, batch_accuracies)[0, 1]
            )

    return size, accuracy, f1, conf_mtx.tolist(), len_acc_corr


def compute_model_size(
        model: mlx.nn.Module,
        size_key_filters: typing.Optional[typing.Dict[str, bool]]=None,
) -> int:
    size_key_filters = (
        size_key_filters if size_key_filters is not None else {}
    )

    return sum(
        params.size
        for path, params in mlx.utils.tree_flatten(model.parameters())
        if all(
            (infix in path) == expectation
            for infix, expectation in size_key_filters.items()
        )
    )


def eval_causal_language_model(
        model: mlx.nn.Module,
        test_batch_fn: typing.Callable[[int], typing.Iterable[tuple]],
        size_key_filters: typing.Optional[typing.Dict[str, bool]]=None,
) -> tuple[int, float]:
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    for X, Y_true in test_batch_fn(0):
        X_len = X.shape[1]
        Y_pred = model(X)
        is_valid = Y_true != TOK_PAD
        losses = mlx.nn.losses.cross_entropy(Y_pred, Y_true, reduction="none")
        losses = losses * is_valid

        mlx.core.eval(losses)

        total_loss += float(mlx.core.sum(losses).item())
        total_tokens += int(mlx.core.sum(is_valid).item())

    size = compute_model_size(model, size_key_filters)

    average_loss = total_loss / max(total_tokens, 1)
    perplexity = math.exp(min(average_loss, 30.0))

    return size, perplexity

### Datasets

#### Utilities

In [22]:
def download_dataset(
        base_url: str,
        files: collections.abc.Iterable[tuple[str, str]],
        target_path: pathlib.Path,
):
    for file_name, sha256 in files:
        download_file(
            base_url + file_name,
            target_path / file_name,
            sha256,
        )


def download_file(
        url: str,
        target: pathlib.Path,
        expected_sha256: typing.Optional[str],
):
    if target.exists():
        print(f"Found {str(target)!r}")
    else:
        one_mb = 1024 * 1024
        buf_size_bytes = 512 * 1024
        target.parent.mkdir(parents=True, exist_ok=True)
    
        print(f"Downloading {url!r}...")
    
        request = urllib.request.Request(url)
        response = urllib.request.urlopen(request)
        next_status_time = 0.0
        total_bytes = int(response.headers.get("Content-Length", "0"))
        total_mb = total_bytes / one_mb
        downloaded_bytes = 0
    
        with response:
            partial = target.with_suffix(target.suffix + ".part")
    
            with partial.open("wb") as f:
                while True:
                    buf = response.read(buf_size_bytes)
    
                    f.write(buf)
    
                    downloaded_bytes += len(buf)
    
                    now = time.time()
    
                    if now > next_status_time or not buf:
                        next_status_time = now + 2.0
                        downloaded_mb = downloaded_bytes / one_mb
    
                        if total_bytes > 0:
                            perc = (downloaded_bytes / total_bytes) * 100.0
                            print(f"  received {downloaded_mb:.1f} MB / {total_mb:.1f} MB ({perc:.2f}%)")
                        else:
                            print(f"  received {downloaded_mb:.1f} MB")
    
                    if not buf:
                        break
    
            partial.replace(target)

    if expected_sha256 is not None:
        print(f"  Verifying {str(target)!r}...", end="")

        actual_sha256 = hashlib.sha256(target.read_bytes()).hexdigest()
    
        if expected_sha256 != actual_sha256:
            raise RuntimeError(
                f"SHA-256 mismatch for {target}: {expected_sha256=!r}, {actual_sha256=!r}"
            )

        print(" OK")

    print("")


def split_train_val(arr: typing.List, val_ratio: float) -> tuple[typing.List, typing.List]:
    indices = list(range(len(arr)))
    random.shuffle(indices)

    val_len = int(len(indices) * val_ratio + 0.5)

    train_indices = sorted(indices[val_len:])
    val_indices = sorted(indices[:val_len])

    train = [arr[i] for i in train_indices]
    val = [arr[i] for i in val_indices]

    return train, val

#### AG's News dataset

In [23]:
AGNEWS_BASE_URL = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/"
AGNEWS_FILES = (
    ("train.csv", "76a0a2d2f92b286371fe4d4044640910a04a803fdd2538e0f3f29a5c6f6b672e"),
    ("test.csv", "521465c2428ed7f02f8d6db6ffdd4b5447c1c701962353eb2c40d548c3c85699"),
)
AGNEWS_PATH = DATA_PATH / "ag-news"


def load_agnews():
    download_dataset(
        AGNEWS_BASE_URL, AGNEWS_FILES, AGNEWS_PATH
    )

    agnews = {}

    for file_name, _ in AGNEWS_FILES:
        split = file_name.replace(".csv", "")
        samples = []

        with (AGNEWS_PATH / file_name).open("r", encoding="utf-8") as f:
            for row in csv.reader(f):
                if len(row) < 3:
                    continue

                label = int(row[0]) - 1
                text = row[1] + " / " + row[2]
                samples.append((text, label))

        agnews[split] = samples

    train, val = split_train_val(agnews["train"], 0.05)

    return train, val, agnews["test"]

#### CIFAR-10 dataset

In [24]:
CIFAR10_BASE_URL = "https://cave.cs.toronto.edu/kriz/"
CIFAR10_FILES = (
    ("cifar-10-binary.tar.gz", "c4a38c50a1bc5f3a1c5537f2155ab9d68f9f25eb1ed8d9ddda3db29a59bca1dd"),
)
CIFAR10_PATH = DATA_PATH / "cifar-10"


def load_cifar10():
    download_dataset(
        CIFAR10_BASE_URL, CIFAR10_FILES, CIFAR10_PATH
    )

    train = []
    test = []

    with tarfile.open(CIFAR10_PATH / CIFAR10_FILES[0][0], "r:*") as tar:
        for member in tar:
            if not (member.isfile() and member.name.endswith(".bin")):
                continue

            is_train = "data_batch_" in member.name
            is_test = "test_batch.bin" in member.name

            if not (is_train or is_test):
                continue

            member_fp = tar.extractfile(member)

            if member_fp is None:
                continue

            raw_samples = (
                np.frombuffer(member_fp.read(), dtype=np.uint8)
                    .reshape(-1, 3073)
            )
            labels = raw_samples[:, 0].astype(int)
            images = (
                raw_samples[:, 1:]
                    .reshape(-1, 3, 32, 32)
                    .transpose(0, 2, 3, 1)
                    .astype(np.float32) / 255.0
            )

            for img, label in zip(images, labels):
                img = mlx.core.array(img)
                sample = (img, int(label))

                if is_train:
                    train.append(sample)
                else:
                    test.append(sample)

    train, val = split_train_val(train, 0.05)

    return train, val, test

#### CIFAR-100 dataset

In [25]:
CIFAR100_BASE_URL = "https://cave.cs.toronto.edu/kriz/"
CIFAR100_FILES = (
    ("cifar-100-binary.tar.gz", "58a81ae192c23a4be8b1804d68e518ed807d710a4eb253b1f2a199162a40d8ec"),
)
CIFAR100_PATH = DATA_PATH / "cifar-100"


def load_cifar100():
    download_dataset(
        CIFAR100_BASE_URL, CIFAR100_FILES, CIFAR100_PATH
    )

    train = []
    test = []

    with tarfile.open(CIFAR100_PATH / CIFAR100_FILES[0][0], "r:*") as tar:
        for member in tar:
            if not member.isfile():
                continue

            is_train = "train.bin" in member.name
            is_test = "test.bin" in member.name

            if not (is_train or is_test):
                continue

            member_fp = tar.extractfile(member)

            if member_fp is None:
                continue

            raw_samples = (
                np.frombuffer(member_fp.read(), dtype=np.uint8)
                    .reshape(-1, 3074)
            )
            coarse_labels = raw_samples[:, 0].astype(int)
            fine_labels = raw_samples[:, 1].astype(int)
            images = (
                raw_samples[:, 2:]
                    .reshape(-1, 3, 32, 32)
                    .transpose(0, 2, 3, 1)
                    .astype(np.float32) / 255.0
            )

            for img, coarse_label, fine_label in zip(images, coarse_labels, fine_labels):
                img = mlx.core.array(img)
                sample = (img, int(coarse_label), int(fine_label))

                if is_train:
                    train.append(sample)
                else:
                    test.append(sample)

    train, val = split_train_val(train, 0.05)

    return train, val, test

#### IMDb dataset

In [26]:
IMDB_BASE_URL = "https://ai.stanford.edu/~amaas/data/sentiment/"
IMDB_FILES = (
    ("aclImdb_v1.tar.gz", "c40f74a18d3b61f90feba1e17730e0d38e8b97c05fde7008942e91923d1658fe"),
)
IMDB_PATH = DATA_PATH / "imdb"


def load_imdb(include_unsupervised: bool):
    download_dataset(
        IMDB_BASE_URL, IMDB_FILES, IMDB_PATH
    )

    reviews = {}
    review_path_re = re.compile(
        r"^aclImdb/(train|test)/(pos|neg"
        + (r"|unsup" if include_unsupervised else r"")
        + r")/([0-9]+)_([0-9]+)\.txt$"
    )

    with tarfile.open(IMDB_PATH / IMDB_FILES[0][0], "r:*") as tar:
        for member in tar:
            if not member.isfile():
                continue

            match = review_path_re.match(member.name)

            if match is None:
                continue

            member_fp = tar.extractfile(member)

            if member_fp is None:
                continue

            split = match[1]
            sentiment = None
            score = None

            if match[2] == "unsup":
                split = "train"
            else:
                sentiment = 1 if match[2] == "pos" else 0
                score = int(match[4])

            text = member_fp.read().decode("utf-8", errors="replace")

            reviews.setdefault(split, []).append((text, sentiment, score))

    train, val = split_train_val(reviews["train"], 0.05)

    return train, val, reviews["test"]

#### ListOps dataset

In [27]:
LISTOPS_BASE_URL = "https://raw.githubusercontent.com/nyu-mll/spinn/refs/heads/listops-release/python/spinn/data/listops/"
LISTOPS_FILES = (
    ("train_d20s.tsv", "655a1ffd18e1044e3bd0b1d891825a8d7c1502ab3dc053452f2491094dbfe65c"),
    ("test_d20s.tsv", "71335416e6896e8dd8dc69281c8ed9b37316382d05d3705d660461564e843840"),
)
LISTOPS_PATH = DATA_PATH / "list-ops"


def load_listops():
    download_dataset(
        LISTOPS_BASE_URL, LISTOPS_FILES, LISTOPS_PATH
    )

    listops = {}

    for file_name, _ in LISTOPS_FILES:
        split = file_name.replace("_d20s.tsv", "")
        samples = []

        with (LISTOPS_PATH / file_name).open("r", encoding="utf-8") as f:
            for line in f:
                row = line.strip().split("\t", 1)
                if len(row) != 2:
                    continue

                label = int(row[0])
                text = row[1]
                samples.append((text, label))

        listops[split] = samples

    train, val = split_train_val(listops["train"], 0.05)

    return train, val, listops["test"]

#### WikiText2 dataset

In [28]:
WIKITEXT2_BASE_URL = "https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/"
WIKITEXT2_FILES = (
    ("train.txt", "9e9fa1ad55b1c2c95b08e37dd8e653f638fac2c6de904b79e813611eefbc985f"),
    ("valid.txt", "f0737ed31fc1329026e95cb8b98e19c2a182c39c240ab909dc31abf2f8af58e8"),
    ("test.txt", "d790b833ef8cf03a90db7bf1271b7520b83c45ce07ba3c1a9699df81e239eca0"),
)
WIKITEXT2_PATH = DATA_PATH / "wikitext2"


def load_wikitext2():
    def parse_articles(path: pathlib.Path) -> typing.List[str]:
        title_re = re.compile(r"^\s*=\s[^=]*\s*=\s*$")
        text = path.read_text()
        article = []

        for line in text.splitlines() + [" = End = "]:
            if title_re.match(line):
                if len(article) > 0:
                    article = "\n".join(article)
                    article = article.strip()

                    if len(article) > 0:
                        yield article

                article = []

            article.append(line)

    download_dataset(
        WIKITEXT2_BASE_URL, WIKITEXT2_FILES, WIKITEXT2_PATH
    )

    train = list(parse_articles(WIKITEXT2_PATH / "train.txt"))
    val = list(parse_articles(WIKITEXT2_PATH / "valid.txt"))
    test = list(parse_articles(WIKITEXT2_PATH / "test.txt"))

    return train, val, test

### Experiments

In [29]:
experiments = {}


def init_experiment(log_path: typing.Optional[pathlib.Path]=None):
    log_msg("Starting experiment...", log_path=log_path)
    set_random_seed(42, log_path=log_path)


def eval_classification_experiment(
        model_name: str,
        test: collections.abc.Sequence,
        snapshot_dir: pathlib.Path,
        model: mlx.nn.Module,
        test_batch_fn: typing.Callable,
        collate_fn: typing.Callable,
        epoch_times: typing.List[tuple[float, float]],
        metrics: dict,
        size_key_filters: typing.Optional[typing.Dict[str, bool]]=None,
        log_path: typing.Optional[pathlib.Path]=None,
):
    log_msg("  Metrics...", log_path=log_path)

    model.eval()

    metrics_replay = Replay(snapshot_dir / "metrics.json.gz")
    size, accuracy, f1, conf_mtx, len_acc_corr = metrics_replay(
        "metrics",
        lambda: eval_classifier(
            model=model,
            test_batch_fn=test_batch_fn,
            size_key_filters=size_key_filters,
        )
    )
    metrics_replay.save()

    train_time = sum(train_time for train_time, _ in epoch_times)

    model_metrics = metrics.setdefault(model_name, {})
    model_metrics.setdefault("size", []).append(size)
    model_metrics.setdefault("accuracy", []).append(accuracy)
    model_metrics.setdefault("f1", []).append(f1)
    model_metrics.setdefault("train_time", []).append(train_time)
    model_metrics.setdefault("len_acc_corr", []).append(len_acc_corr)

    log_msg(f"    Model size: {size:_}", log_path=log_path)
    log_msg(
        f"      Accuracy: {accuracy:.5f}  (len corr: {len_acc_corr:.3f})",
        log_path=log_path,
    )
    log_msg(f"            F1: {f1:.5f}", log_path=log_path)
    log_msg(f"    Train time: {train_time:.1f}s", log_path=log_path)
    log_msg("", log_path=log_path)

    if DEBUG:
        print(f"  Confusion matrix:")
        cm_disp = sklearn.metrics.ConfusionMatrixDisplay(
            np.array(conf_mtx)
        )
        cm_disp.plot()
        plt.show()
        print("")

        test_indices = list(range(len(test)))
        random.shuffle(test_indices)
        test_indices = test_indices[:10]
    
        for i in test_indices:
            X, Y_true = collate_fn([test[i]])
            Y_pred = model(X)
    
            if Y_pred.shape[1] <= 1:
                Y_pred = mlx.core.sigmoid(Y_pred)
            else:
                Y_pred = mlx.core.softmax(Y_pred)
    
            Y = mlx.core.stack([Y_pred[0], Y_true[0]]).T
    
            for yp, yt in Y.tolist():
                print(f"{int(yt)}:{yp:.3f},  ", end="")
    
            print("")
    
            sample_input = test[i][0]
    
            if isinstance(sample_input, str):
                print(sample_input)
            else:
                cmap = "gray" if sample_input.shape[2] == 1 else None
                plt.figure(figsize=(1.2, 1.2))
                plt.imshow(sample_input, cmap=cmap)
                plt.show()
    
            print("")


def print_classification_experiments_summary(metrics):
    print_experiments_summary(
        metrics,
        ("size", "accuracy", "f1", "train_time", "len_acc_corr"),
    )


def print_experiments_summary(metrics, metric_names):
    print("")
    print("Metrics")

    for metric in metric_names:
        print(f"  {metric.title()}:")

        for model_name, model_metrics in metrics.items():
            model_metric = mlx.core.array(model_metrics[metric])

            if metric == "size":
                print(f"    {model_name:>42}: {model_metric[0]:_}")
            else:
                print(
                    f"    {model_name:>42}:"
                    f" {model_metric.mean().item():.5f}"
                    f" +/- {model_metric.std().item():.5f}"
                )

        print("")


def compute_seq_lens(
        dataset: typing.List,
        collate_fn: typing.Callable=collate,
) -> tuple[float, float]:
    seq_lens = mlx.core.array(
        [collate_fn([sample])[0].shape[1] for sample in dataset],
        dtype=mlx.core.float32,
    )

    mean = float(seq_lens.mean().item())
    std = float(seq_lens.std().item())

    return mean, std


def eval_causal_language_model_experiment(
        model_name: str,
        test: collections.abc.Sequence,
        snapshot_dir: pathlib.Path,
        model: mlx.nn.Module,
        vocabulary: Vocabulary,
        test_batch_fn: typing.Callable,
        epoch_times: typing.List[tuple[float, float]],
        metrics: dict,
        size_key_filters: typing.Optional[typing.Dict[str, bool]]=None,
        log_path: typing.Optional[pathlib.Path]=None,
):
    log_msg("  Metrics...", log_path=log_path)

    model.eval()

    metrics_replay = Replay(snapshot_dir / "metrics.json.gz")
    size, perplexity = metrics_replay(
        "metrics",
        lambda: eval_causal_language_model(
            model=model,
            test_batch_fn=test_batch_fn,
            size_key_filters=size_key_filters,
        )
    )
    metrics_replay.save()

    train_time = sum(train_time for train_time, _ in epoch_times)

    model_metrics = metrics.setdefault(model_name, {})
    model_metrics.setdefault("size", []).append(size)
    model_metrics.setdefault("perplexity", []).append(perplexity)
    model_metrics.setdefault("train_time", []).append(train_time)

    log_msg(f"    Model size: {size:_}", log_path=log_path)
    log_msg(f"    Perplexity: {perplexity:.5f}", log_path=log_path)
    log_msg(f"    Train time: {train_time:.1f}s", log_path=log_path)
    log_msg("", log_path=log_path)

    if DEBUG:
        prompts = " The A An For Once In How What When Therefore Not".split(" ")
        
        for i in range(10):
            prompt = random.choice(prompts)
            generated = generate_text(
                model=model,
                vocabulary=vocabulary,
                prompt=prompt,
                max_length=256,
                temperature=1.0,
            )
            print(f"##### Example {i+1}:")
            print(prompt + " " + generated)
            print("")


def print_causal_language_models_experiments_summary(metrics):
    print_experiments_summary(
        metrics,
        ("size", "perplexity", "train_time"),
    )

#### AG's News experiment

In [30]:
def run_agnews():
    log_path = RESULTS_PATH / "agnews.txt"
    num_epochs = 1
    num_classes = 4
    num_layers = 6
    dims = 72
    num_heads = 3
    init_experiment(log_path=log_path)
    train, val, test = load_agnews()
    vocabulary = Vocabulary(texts=(text for text, _ in train))
    common_params = dict(
        vocabulary_size=len(vocabulary),
        num_layers=num_layers,
        dims=dims,
        dims_ffn=4 * dims,
        num_classes=num_classes,
    )
    softmax = lambda: TextClassifier(
        num_heads=num_heads,
        attention_type=AttentionType.SOFTMAX,
        **common_params,
    )
    relu = lambda: TextClassifier(
        num_heads=num_heads,
        attention_type=AttentionType.RELU,
        **common_params,
    )
    relu_len_scale = lambda: TextClassifier(
        num_heads=num_heads,
        attention_type=AttentionType.RELU,
        scaling=AttentionScaling.SEQ_LEN,
        **common_params,
    )
    simplified_relu = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.SIMPLIFIED_RELU,
        **common_params,
    )
    simplified_relu_no_scale = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.SIMPLIFIED_RELU,
        scaling=AttentionScaling.NONE,
        **common_params,
    )
    simplified_relu_len_scale = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.SIMPLIFIED_RELU,
        scaling=AttentionScaling.SEQ_LEN,
        **common_params,
    )
    unbiased_perceptron = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.UNBIASED_PERCEPTRON,
        scaling=AttentionScaling.NONE,
        **common_params,
    )
    unbiased_perceptron_len_scale = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.UNBIASED_PERCEPTRON,
        scaling=AttentionScaling.SEQ_LEN,
        **common_params,
    )
    perceptron = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.PERCEPTRON,
        scaling=AttentionScaling.NONE,
        **common_params,
    )
    perceptron_len_scale = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.PERCEPTRON,
        scaling=AttentionScaling.SEQ_LEN,
        **common_params,
    )
    collate_fn = (
        lambda samples: collate_text_classification_samples(
            vocabulary, samples, num_classes
        )
    )

    batch_size = 8
    num_grad_acc_batches = 4
    random_seed_base = 424242
    train_batch_fn_basic = lambda epoch: generate_batches(
        epoch=epoch,
        samples=train,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
    )
    train_batch_fn_drop_in = lambda epoch: generate_batches_drop_in(
        model=model, # Late binding!
        epoch=epoch,
        train_batch_fn=train_batch_fn_basic,
        drop_in_schedule=(
            (0,     0, 0.2),
            (0,  2000, 0.3),
            (0,  3400, 0.4),
            (0,  4800, 0.5),
            (0,  6200, 0.6),
            (0,  7600, 0.7),
            (0,  9000, 0.8),
            (0, 10400, 0.9),
            (0, 11800, 1.0),
        ),
        log_path=log_path,
    )
    val_batch_fn = lambda epoch: generate_batches(
        epoch=epoch,
        samples=val,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )
    test_batch_fn = lambda epoch: generate_batches(
        epoch=epoch,
        samples=test,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )

    models = (
        ("Softmax", "ag-news-softmax", train_batch_fn_basic, softmax),
        ("ReLU", "ag-news-relu", train_batch_fn_basic, relu),
        ("ReLU/seql", "ag-news-relu-seql", train_batch_fn_basic, relu_len_scale),
        ("SReLU", "ag-news-simplified-relu", train_batch_fn_basic, simplified_relu),
        ("SReLU/noscl", "ag-news-simplified-relu-no-scale", train_batch_fn_basic, simplified_relu_no_scale),
        ("SReLU/seql", "ag-news-simplified-relu-seql", train_batch_fn_basic, simplified_relu_len_scale),
        ("UbPerc", "ag-news-unbiased-perceptron", train_batch_fn_basic, unbiased_perceptron),
        ("UbPerc/seql", "ag-news-unbiased-perceptron-seql", train_batch_fn_basic, unbiased_perceptron_len_scale),
        ("Perc", "ag-news-perceptron", train_batch_fn_basic, perceptron),
        ("Perc/seql", "ag-news-perceptron-seql", train_batch_fn_basic, perceptron_len_scale),
        ("Softmax+DI", "ag-news-softmax-drop-in", train_batch_fn_drop_in, softmax),
        ("ReLU+DI", "ag-news-relu-drop-in", train_batch_fn_drop_in, relu),
        ("ReLU/seql+DI", "ag-news-relu-seql-drop-in", train_batch_fn_drop_in, relu_len_scale),
        ("SReLU+DI", "ag-news-simplified-relu-drop-in", train_batch_fn_drop_in, simplified_relu),
        ("SReLU/noscl+DI", "ag-news-simplified-relu-no-scale-drop-in", train_batch_fn_drop_in, simplified_relu_no_scale),
        ("SReLU/seql+DI", "ag-news-simplified-relu-seql-drop-in", train_batch_fn_drop_in, simplified_relu_len_scale),
        ("UbPerc+DI", "ag-news-unbiased-perceptron-drop-in", train_batch_fn_drop_in, unbiased_perceptron),
        ("UbPerc/seql+DI", "ag-news-unbiased-perceptron-seql-drop-in", train_batch_fn_drop_in, unbiased_perceptron_len_scale),
        ("Perc+DI", "ag-news-perceptron-drop-in", train_batch_fn_drop_in, perceptron),
        ("Perc/seql+DI", "ag-news-perceptron-seql-drop-in", train_batch_fn_drop_in, perceptron_len_scale),
    )

    metrics = {}

    for model_name, snapshot_dir_base, train_batch_fn, make_model in models:
        for i in range(3):
            log_msg(f"AG's News / {model_name} ({i + 1})", log_path=log_path)
            set_random_seed(12345 + 42 * (i + 1), log_path=log_path)
            model = make_model()
            snapshot_dir = MODELS_PATH / f"{snapshot_dir_base}-{i}"
            _, epoch_times = run_training(
                model,
                loss_fn=cross_entropy_loss,
                train_batch_fn=train_batch_fn,
                test_batch_fn=val_batch_fn,
                num_epochs=num_epochs,
                num_grad_acc_batches=num_grad_acc_batches,
                num_status_batches=300,
                snapshot_dir=snapshot_dir,
                num_snapshot_batches=0,
                lr=5e-4,
                log_path=log_path,
            )
            eval_classification_experiment(
                model_name=model_name,
                test=test,
                snapshot_dir=snapshot_dir,
                model=model,
                test_batch_fn=test_batch_fn,
                collate_fn=collate_fn,
                size_key_filters={".indices": False, "embedding.": False},
                epoch_times=epoch_times,
                metrics=metrics,
                log_path=log_path,
            )

    print_classification_experiments_summary(metrics)

    seq_len_mean, seq_len_std = compute_seq_lens(test, collate_fn)

    return metrics, (num_layers, dims, num_heads, seq_len_mean, seq_len_std)

#### CIFAR-10 experiment

In [31]:
def run_cifar10():
    log_path = RESULTS_PATH / "cifar10.txt"
    num_epochs = 100
    num_classes = 10
    num_layers = 12
    dims = 72
    num_heads = 3
    patch_size = 4
    init_experiment(log_path=log_path)
    train, val, test = load_cifar10()
    common_params = dict(
        in_channels=3,
        patch_size=patch_size,
        num_layers=num_layers,
        dims=dims,
        dims_ffn=4 * dims,
        num_classes=num_classes,
    )
    softmax = lambda: ImageClassifier(
        num_heads=num_heads,
        attention_type=AttentionType.SOFTMAX,
        **common_params,
    )
    relu = lambda: ImageClassifier(
        num_heads=num_heads,
        attention_type=AttentionType.RELU,
        **common_params,
    )
    simplified_relu = lambda: ImageClassifier(
        num_heads=1,
        attention_type=AttentionType.SIMPLIFIED_RELU,
        **common_params,
    )
    simplified_relu_no_scale = lambda: ImageClassifier(
        num_heads=1,
        attention_type=AttentionType.SIMPLIFIED_RELU,
        scaling=AttentionScaling.NONE,
        **common_params,
    )
    unbiased_perceptron = lambda: ImageClassifier(
        num_heads=1,
        attention_type=AttentionType.UNBIASED_PERCEPTRON,
        scaling=AttentionScaling.NONE,
        **common_params,
    )
    perceptron = lambda: ImageClassifier(
        num_heads=1,
        attention_type=AttentionType.PERCEPTRON,
        scaling=AttentionScaling.NONE,
        **common_params,
    )
    collate_fn = (
        lambda samples: collate_image_classification_samples(
            samples, num_classes
        )
    )
    batch_size = 128
    num_grad_acc_batches = 4
    random_seed_base = 424242
    train_batch_fn_basic = lambda epoch: generate_batches(
        epoch=epoch,
        samples=train,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
    )
    train_batch_fn_drop_in = lambda epoch: generate_batches_drop_in(
        model=model, # Late binding!
        epoch=epoch,
        train_batch_fn=train_batch_fn_basic,
        drop_in_schedule=(
            ( 0, 0, 0.2),
            (28, 0, 0.3),
            (35, 0, 0.4),
            (42, 0, 0.5),
            (49, 0, 0.6),
            (56, 0, 0.7),
            (63, 0, 0.8),
            (70, 0, 0.9),
            (77, 0, 1.0),
        ),
        log_path=log_path,
    )
    val_batch_fn = lambda epoch: generate_batches(
        epoch=epoch,
        samples=val,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )
    test_batch_fn = lambda epoch: generate_batches(
        epoch=epoch,
        samples=test,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )

    models = (
        ("Softmax", "cifar10-softmax", train_batch_fn_basic, softmax),
        ("ReLU", "cifar10-relu", train_batch_fn_basic, relu),
        ("SReLU", "cifar10-simplified-relu", train_batch_fn_basic, simplified_relu),
        ("SReLU/noscl", "cifar10-simplified-relu-no-scale", train_batch_fn_basic, simplified_relu_no_scale),
        ("UbPerc", "cifar10-unbiased-perceptron", train_batch_fn_basic, unbiased_perceptron),
        ("Perc", "cifar10-perceptron", train_batch_fn_basic, perceptron),
        ("Softmax+DI", "cifar10-softmax-drop-in", train_batch_fn_drop_in, softmax),
        ("ReLU+DI", "cifar10-relu-drop-in", train_batch_fn_drop_in, relu),
        ("SReLU+DI", "cifar10-simplified-relu-drop-in", train_batch_fn_drop_in, simplified_relu),
        ("SReLU/noscl+DI", "cifar10-simplified-relu-no-scale-drop-in", train_batch_fn_drop_in, simplified_relu_no_scale),
        ("UbPerc+DI", "cifar10-unbiased-perceptron-drop-in", train_batch_fn_drop_in, unbiased_perceptron),
        ("Perc+DI", "cifar10-perceptron-drop-in", train_batch_fn_drop_in, perceptron),
    )

    metrics = {}

    for model_name, snapshot_dir_base, train_batch_fn, make_model in models:
        for i in range(3):
            log_msg(f"CIFAR-10 / {model_name} ({i + 1})", log_path=log_path)
            set_random_seed(12345 + 42 * (i + 1), log_path=log_path)
            model = make_model()
            snapshot_dir = MODELS_PATH / f"{snapshot_dir_base}-{i}"
            _, epoch_times = run_training(
                model,
                loss_fn=cross_entropy_loss,
                train_batch_fn=train_batch_fn,
                test_batch_fn=val_batch_fn,
                num_epochs=num_epochs,
                num_grad_acc_batches=num_grad_acc_batches,
                num_status_batches=100,
                snapshot_dir=snapshot_dir,
                num_snapshot_batches=0,
                lr=1e-3,
                train_aug_fn=lambda X, Y: (randomize_image(X), Y),
                log_path=log_path,
            )
            eval_classification_experiment(
                model_name=model_name,
                test=test,
                snapshot_dir=snapshot_dir,
                model=model,
                test_batch_fn=test_batch_fn,
                collate_fn=collate_fn,
                size_key_filters={".indices": False},
                epoch_times=epoch_times,
                metrics=metrics,
                log_path=log_path,
            )

    print_classification_experiments_summary(metrics)

    seq_len_mean, seq_len_std = (32.0 / patch_size) ** 2.0 + 1.0, 0.0

    return metrics, (num_layers, dims, num_heads, seq_len_mean, seq_len_std)

#### CIFAR-100 experiment

In [32]:
def run_cifar100():
    log_path = RESULTS_PATH / "cifar100.txt"
    num_epochs = 50
    num_classes = 20
    num_layers = 12
    dims = 128
    num_heads = 4
    patch_size = 4
    init_experiment(log_path=log_path)
    train, val, test = load_cifar100()
    common_params = dict(
        in_channels=3,
        patch_size=patch_size,
        num_layers=num_layers,
        dims=dims,
        dims_ffn=4 * dims,
        num_classes=num_classes,
    )
    softmax = lambda: ImageClassifier(
        num_heads=num_heads,
        attention_type=AttentionType.SOFTMAX,
        **common_params,
    )
    relu = lambda: ImageClassifier(
        num_heads=num_heads,
        attention_type=AttentionType.RELU,
        **common_params,
    )
    simplified_relu = lambda: ImageClassifier(
        num_heads=1,
        attention_type=AttentionType.SIMPLIFIED_RELU,
        **common_params,
    )
    simplified_relu_no_scale = lambda: ImageClassifier(
        num_heads=1,
        attention_type=AttentionType.SIMPLIFIED_RELU,
        scaling=AttentionScaling.NONE,
        **common_params,
    )
    unbiased_perceptron = lambda: ImageClassifier(
        num_heads=1,
        attention_type=AttentionType.UNBIASED_PERCEPTRON,
        scaling=AttentionScaling.NONE,
        **common_params,
    )
    perceptron = lambda: ImageClassifier(
        num_heads=1,
        attention_type=AttentionType.PERCEPTRON,
        scaling=AttentionScaling.NONE,
        **common_params,
    )
    collate_fn = (
        lambda samples: collate_image_classification_samples(
            ((img, coarse_label) for img, coarse_label, _ in samples),
            num_classes
        )
    )
    batch_size = 128
    num_grad_acc_batches = 4
    random_seed_base = 424242
    train_batch_fn_basic = lambda epoch: generate_batches(
        epoch=epoch,
        samples=train,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
    )
    train_batch_fn_drop_in = lambda epoch: generate_batches_drop_in(
        model=model, # Late binding!
        epoch=epoch,
        train_batch_fn=train_batch_fn_basic,
        drop_in_schedule=(
            ( 0, 0, 0.2),
            (15, 0, 0.3),
            (18, 0, 0.4),
            (21, 0, 0.5),
            (24, 0, 0.6),
            (27, 0, 0.7),
            (30, 0, 0.8),
            (33, 0, 0.9),
            (36, 0, 1.0),
        ),
        log_path=log_path,
    )
    val_batch_fn = lambda epoch: generate_batches(
        epoch=epoch,
        samples=val,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )
    test_batch_fn = lambda epoch: generate_batches(
        epoch=epoch,
        samples=test,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )

    models = (
        ("Softmax", "cifar100-softmax", train_batch_fn_basic, softmax),
        ("ReLU", "cifar100-relu", train_batch_fn_basic, relu),
        ("SReLU", "cifar100-simplified-relu", train_batch_fn_basic, simplified_relu),
        ("SReLU/noscl", "cifar100-simplified-relu-no-scale", train_batch_fn_basic, simplified_relu_no_scale),
        ("UbPerc", "cifar100-unbiased-perceptron", train_batch_fn_basic, unbiased_perceptron),
        ("Perc", "cifar100-perceptron", train_batch_fn_basic, perceptron),
        ("Softmax+DI", "cifar100-softmax-drop-in", train_batch_fn_drop_in, softmax),
        ("ReLU+DI", "cifar100-relu-drop-in", train_batch_fn_drop_in, relu),
        ("SReLU+DI", "cifar100-simplified-relu-drop-in", train_batch_fn_drop_in, simplified_relu),
        ("SReLU/noscl+DI", "cifar100-simplified-relu-no-scale-drop-in", train_batch_fn_drop_in, simplified_relu_no_scale),
        ("UbPerc+DI", "cifar100-unbiased-perceptron-drop-in", train_batch_fn_drop_in, unbiased_perceptron),
        ("Perc+DI", "cifar100-perceptron-drop-in", train_batch_fn_drop_in, perceptron),
    )

    metrics = {}

    for model_name, snapshot_dir_base, train_batch_fn, make_model in models:
        for i in range(3):
            log_msg(f"CIFAR-100 / {model_name} ({i + 1})", log_path=log_path)
            set_random_seed(12345 + 42 * (i + 1), log_path=log_path)
            model = make_model()
            snapshot_dir = MODELS_PATH / f"{snapshot_dir_base}-{i}"
            _, epoch_times = run_training(
                model,
                loss_fn=cross_entropy_loss,
                train_batch_fn=train_batch_fn,
                test_batch_fn=val_batch_fn,
                num_epochs=num_epochs,
                num_grad_acc_batches=num_grad_acc_batches,
                num_status_batches=100,
                snapshot_dir=snapshot_dir,
                num_snapshot_batches=0,
                lr=1e-3,
                train_aug_fn=lambda X, Y: (randomize_image(X), Y),
                log_path=log_path,
            )
            eval_classification_experiment(
                model_name=model_name,
                test=test,
                snapshot_dir=snapshot_dir,
                model=model,
                test_batch_fn=test_batch_fn,
                collate_fn=collate_fn,
                size_key_filters={".indices": False},
                epoch_times=epoch_times,
                metrics=metrics,
                log_path=log_path,
            )

    print_classification_experiments_summary(metrics)

    seq_len_mean, seq_len_std = (32.0 / patch_size) ** 2.0 + 1.0, 0.0

    return metrics, (num_layers, dims, num_heads, seq_len_mean, seq_len_std)

#### IMDb experiment

In [33]:
def run_imdb():
    log_path = RESULTS_PATH / "imdb.txt"
    num_epochs = 1
    num_classes = 1
    num_layers = 9
    dims = 144
    num_heads = 6
    init_experiment(log_path=log_path)
    train, val, test = load_imdb(include_unsupervised=False)
    sample_num_chars_max = 4500
    train = [(s, l) for s, l, _ in train if len(s) < sample_num_chars_max]
    val = [(s, l) for s, l, _ in val if len(s) < sample_num_chars_max]
    test = [(s, l) for s, l, _ in test if len(s) < sample_num_chars_max]
    log_msg(f"{len(train)=}, {len(val)=}, {len(test)=}", log_path=log_path)
    vocabulary = Vocabulary(texts=(text for text, _ in train))
    common_params = dict(
        vocabulary_size=len(vocabulary),
        num_layers=num_layers,
        dims=dims,
        dims_ffn=4 * dims,
        num_classes=num_classes,
    )
    softmax = lambda: TextClassifier(
        num_heads=num_heads,
        attention_type=AttentionType.SOFTMAX,
        **common_params,
    )
    relu = lambda: TextClassifier(
        num_heads=num_heads,
        attention_type=AttentionType.RELU,
        **common_params,
    )
    relu_len_scale = lambda: TextClassifier(
        num_heads=num_heads,
        attention_type=AttentionType.RELU,
        scaling=AttentionScaling.SEQ_LEN,
        **common_params,
    )
    simplified_relu = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.SIMPLIFIED_RELU,
        **common_params,
    )
    simplified_relu_no_scale = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.SIMPLIFIED_RELU,
        scaling=AttentionScaling.NONE,
        **common_params,
    )
    simplified_relu_len_scale = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.SIMPLIFIED_RELU,
        scaling=AttentionScaling.SEQ_LEN,
        **common_params,
    )
    unbiased_perceptron = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.UNBIASED_PERCEPTRON,
        scaling=AttentionScaling.NONE,
        **common_params,
    )
    unbiased_perceptron_len_scale = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.UNBIASED_PERCEPTRON,
        scaling=AttentionScaling.SEQ_LEN,
        **common_params,
    )
    perceptron = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.PERCEPTRON,
        scaling=AttentionScaling.NONE,
        **common_params,
    )
    perceptron_len_scale = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.PERCEPTRON,
        scaling=AttentionScaling.SEQ_LEN,
        **common_params,
    )
    collate_fn = (
        lambda samples: collate_text_classification_samples(
            vocabulary, samples, num_classes
        )
    )
    batch_size = 8
    num_grad_acc_batches = 4
    random_seed_base = 424242
    train_batch_fn_basic = lambda epoch: generate_batches(
        epoch=epoch,
        samples=train,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
    )
    train_batch_fn_drop_in = lambda epoch: generate_batches_drop_in(
        model=model, # Late binding!
        epoch=epoch,
        train_batch_fn=train_batch_fn_basic,
        drop_in_schedule=(
            (0,    0, 0.2),
            (0,  600, 0.3),
            (0,  800, 0.4),
            (0, 1000, 0.5),
            (0, 1200, 0.6),
            (0, 1400, 0.7),
            (0, 1600, 0.8),
            (0, 1800, 0.9),
            (0, 2000, 1.0),
        ),
        log_path=log_path,
    )
    val_batch_fn = lambda epoch: generate_batches(
        epoch=epoch,
        samples=val,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )
    test_batch_fn = lambda epoch: generate_batches(
        epoch=epoch,
        samples=test,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )

    models = (
        ("Softmax", "imdb-softmax", train_batch_fn_basic, softmax),
        ("ReLU", "imdb-relu", train_batch_fn_basic, relu),
        ("ReLU/seql", "imdb-relu-seql", train_batch_fn_basic, relu_len_scale),
        ("SReLU", "imdb-simplified-relu", train_batch_fn_basic, simplified_relu),
        ("SReLU/noscl", "imdb-simplified-relu-no-scale", train_batch_fn_basic, simplified_relu_no_scale),
        ("SReLU/seql", "imdb-simplified-relu-seql", train_batch_fn_basic, simplified_relu_len_scale),
        ("UbPerc", "imdb-unbiased-perceptron", train_batch_fn_basic, unbiased_perceptron),
        ("UbPerc/seql", "imdb-unbiased-perceptron-seql", train_batch_fn_basic, unbiased_perceptron_len_scale),
        ("Perc", "imdb-perceptron", train_batch_fn_basic, perceptron),
        ("Perc/seql", "imdb-perceptron-seql", train_batch_fn_basic, perceptron_len_scale),
        ("Softmax+DI", "imdb-softmax-drop-in", train_batch_fn_drop_in, softmax),
        ("ReLU+DI", "imdb-relu-drop-in", train_batch_fn_drop_in, relu),
        ("ReLU/seql+DI", "imdb-relu-seql-drop-in", train_batch_fn_drop_in, relu_len_scale),
        ("SReLU+DI", "imdb-simplified-relu-drop-in", train_batch_fn_drop_in, simplified_relu),
        ("SReLU/noscl+DI", "imdb-simplified-relu-no-scale-drop-in", train_batch_fn_drop_in, simplified_relu_no_scale),
        ("SReLU/seql+DI", "imdb-simplified-relu-seql-drop-in", train_batch_fn_drop_in, simplified_relu_len_scale),
        ("UbPerc+DI", "imdb-unbiased-perceptron-drop-in", train_batch_fn_drop_in, unbiased_perceptron),
        ("UbPerc/seql+DI", "imdb-unbiased-perceptron-seql-drop-in", train_batch_fn_drop_in, unbiased_perceptron_len_scale),
        ("Perc+DI", "imdb-perceptron-drop-in", train_batch_fn_drop_in, perceptron),
        ("Perc/seql+DI", "imdb-perceptron-seql-drop-in", train_batch_fn_drop_in, perceptron_len_scale),
    )

    metrics = {}

    for model_name, snapshot_dir_base, train_batch_fn, make_model in models:
        for i in range(3):
            log_msg(f"IMDb / {model_name} ({i + 1})", log_path=log_path)
            set_random_seed(12345 + 42 * (i + 1), log_path=log_path)
            model = make_model()
            snapshot_dir = MODELS_PATH / f"{snapshot_dir_base}-{i}"
            _, epoch_times = run_training(
                model,
                loss_fn=binary_cross_entropy_loss,
                train_batch_fn=train_batch_fn,
                test_batch_fn=val_batch_fn,
                num_epochs=num_epochs,
                num_grad_acc_batches=num_grad_acc_batches,
                num_status_batches=100,
                snapshot_dir=snapshot_dir,
                num_snapshot_batches=1000,
                lr=5e-4,
                log_path=log_path,
            )
            eval_classification_experiment(
                model_name=model_name,
                test=test,
                snapshot_dir=snapshot_dir,
                model=model,
                test_batch_fn=test_batch_fn,
                collate_fn=collate_fn,
                size_key_filters={".indices": False, "embedding.": False},
                epoch_times=epoch_times,
                metrics=metrics,
                log_path=log_path,
            )

    print_classification_experiments_summary(metrics)

    seq_len_mean, seq_len_std = compute_seq_lens(test, collate_fn)

    return metrics, (num_layers, dims, num_heads, seq_len_mean, seq_len_std)

#### IMDb-gen experiment

In [34]:
def run_imdbgen():
    log_path = RESULTS_PATH / "imdbgen.txt"
    num_epochs = 1
    seq_len = 256
    num_layers = 12
    dims = 256
    num_heads = 8
    init_experiment(log_path=log_path)
    train, val, test = load_imdb(include_unsupervised=True)
    train = [text for text, _, _ in train]
    val = [text for text, _, _ in val]
    test = [text for text, _, _ in test]
    vocabulary = Vocabulary(texts=train)
    common_params = dict(
        vocabulary_size=len(vocabulary),
        num_layers=num_layers,
        dims=dims,
        dims_ffn=4 * dims,
    )
    softmax = lambda: CausalLanguageModel(
        num_heads=num_heads,
        attention_type=AttentionType.SOFTMAX,
        **common_params,
    )
    relu = lambda: CausalLanguageModel(
        num_heads=num_heads,
        attention_type=AttentionType.RELU,
        **common_params,
    )
    relu_len_scale = lambda: CausalLanguageModel(
        num_heads=num_heads,
        attention_type=AttentionType.RELU,
        scaling=AttentionScaling.SEQ_LEN,
        **common_params,
    )
    simplified_relu = lambda: CausalLanguageModel(
        num_heads=1,
        attention_type=AttentionType.SIMPLIFIED_RELU,
        **common_params,
    )
    simplified_relu_no_scale = lambda: CausalLanguageModel(
        num_heads=1,
        attention_type=AttentionType.SIMPLIFIED_RELU,
        scaling=AttentionScaling.NONE,
        **common_params,
    )
    simplified_relu_len_scale = lambda: CausalLanguageModel(
        num_heads=1,
        attention_type=AttentionType.SIMPLIFIED_RELU,
        scaling=AttentionScaling.SEQ_LEN,
        **common_params,
    )

    batch_size = 8
    num_grad_acc_batches = 32
    random_seed_base = 424242
    train_batch_fn_basic = lambda epoch: generate_causal_lm_batches(
        epoch=epoch,
        texts=train,
        seq_len=seq_len,
        vocabulary=vocabulary,
        batch_size=batch_size,
        shuffle=True,
    )
    train_batch_fn_drop_in = lambda epoch: generate_batches_drop_in(
        model=model, # Late binding!
        epoch=epoch,
        train_batch_fn=train_batch_fn_basic,
        drop_in_schedule=(
            (0,     0, 0.2),
            (0,  3000, 0.3),
            (0,  4000, 0.4),
            (0,  5000, 0.5),
            (0,  6000, 0.6),
            (0,  7000, 0.7),
            (0,  8000, 0.8),
            (0,  9000, 0.9),
            (0, 10000, 1.0),
        ),
        log_path=log_path,
    )
    val_batch_fn = lambda epoch: generate_causal_lm_batches(
        epoch=epoch,
        texts=val,
        seq_len=seq_len,
        vocabulary=vocabulary,
        batch_size=batch_size,
        shuffle=False,
    )
    test_batch_fn = lambda epoch: generate_causal_lm_batches(
        epoch=epoch,
        texts=test,
        seq_len=seq_len,
        vocabulary=vocabulary,
        batch_size=batch_size,
        shuffle=False,
    )

    models = (
        ("Softmax", "imdbgen-softmax", train_batch_fn_basic, softmax),
        ("ReLU", "imdbgen-relu", train_batch_fn_basic, relu),
        ("ReLU/seql", "imdbgen-relu-seql", train_batch_fn_basic, relu_len_scale),
        ("SReLU", "imdbgen-simplified-relu", train_batch_fn_basic, simplified_relu),
        ("SReLU/noscl", "imdbgen-simplified-relu-no-scale", train_batch_fn_basic, simplified_relu_no_scale),
        ("SReLU/seql", "imdbgen-simplified-relu-seql", train_batch_fn_basic, simplified_relu_len_scale),
        ("Softmax+DI", "imdbgen-softmax-drop-in", train_batch_fn_drop_in, softmax),
        ("ReLU+DI", "imdbgen-relu-drop-in", train_batch_fn_drop_in, relu),
        ("ReLU/seql+DI", "imdbgen-relu-seql-drop-in", train_batch_fn_drop_in, relu_len_scale),
        ("SReLU+DI", "imdbgen-simplified-relu-drop-in", train_batch_fn_drop_in, simplified_relu),
        ("SReLU/noscl+DI", "imdbgen-simplified-relu-no-scale-drop-in", train_batch_fn_drop_in, simplified_relu_no_scale),
        ("SReLU/seql+DI", "imdbgen-simplified-relu-seql-drop-in", train_batch_fn_drop_in, simplified_relu_len_scale),
    )

    metrics = {}

    for model_name, snapshot_dir_base, train_batch_fn, make_model in models:
        for i in range(3):
            log_msg(f"IMDB-gen / {model_name} ({i + 1})", log_path=log_path)
            set_random_seed(12345 + 42 * (i + 1), log_path=log_path)
            model = make_model()
            snapshot_dir = MODELS_PATH / f"{snapshot_dir_base}-{i}"
            _, epoch_times = run_training(
                model,
                loss_fn=valid_cross_entropy_loss,
                train_batch_fn=train_batch_fn,
                test_batch_fn=val_batch_fn,
                num_epochs=num_epochs,
                num_grad_acc_batches=num_grad_acc_batches,
                num_status_batches=50,
                snapshot_dir=snapshot_dir,
                num_snapshot_batches=0,
                lr=5e-4,
                log_path=log_path,
            )
            eval_causal_language_model_experiment(
                model_name=model_name,
                test=test,
                snapshot_dir=snapshot_dir,
                model=model,
                vocabulary=vocabulary,
                test_batch_fn=test_batch_fn,
                epoch_times=epoch_times,
                metrics=metrics,
                size_key_filters={
                    ".indices": False,
                    "embedding.": False,
                    "output.": False,
                },
                log_path=log_path,
            )

    print_causal_language_models_experiments_summary(metrics)

    seq_len_mean, seq_len_std = float(seq_len), 0.0

    return metrics, (num_layers, dims, num_heads, seq_len_mean, seq_len_std)

#### ListOps experiment

In [35]:
def run_listops():
    log_path = RESULTS_PATH / "listops.txt"
    num_epochs = 2
    num_classes = 10
    num_layers = 12
    dims = 144
    num_heads = 6
    init_experiment(log_path=log_path)
    train, val, test = load_listops()
    sample_num_chars_max = 1000
    train = [(s, l) for s, l in train if len(s) < sample_num_chars_max]
    val = [(s, l) for s, l in val if len(s) < sample_num_chars_max]
    test = [(s, l) for s, l in test if len(s) < sample_num_chars_max]
    log_msg(f"{len(train)=}, {len(val)=}, {len(test)=}", log_path=log_path)
    vocabulary = Vocabulary(texts=(text for text, _ in train))
    common_params = dict(
        vocabulary_size=len(vocabulary),
        num_layers=num_layers,
        dims=dims,
        dims_ffn=4 * dims,
        num_classes=num_classes,
    )
    softmax = lambda: TextClassifier(
        num_heads=num_heads,
        attention_type=AttentionType.SOFTMAX,
        **common_params,
    )
    relu = lambda: TextClassifier(
        num_heads=num_heads,
        attention_type=AttentionType.RELU,
        **common_params,
    )
    relu_len_scale = lambda: TextClassifier(
        num_heads=num_heads,
        attention_type=AttentionType.RELU,
        scaling=AttentionScaling.SEQ_LEN,
        **common_params,
    )
    simplified_relu = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.SIMPLIFIED_RELU,
        **common_params,
    )
    simplified_relu_no_scale = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.SIMPLIFIED_RELU,
        scaling=AttentionScaling.NONE,
        **common_params,
    )
    simplified_relu_len_scale = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.SIMPLIFIED_RELU,
        scaling=AttentionScaling.SEQ_LEN,
        **common_params,
    )
    unbiased_perceptron = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.UNBIASED_PERCEPTRON,
        scaling=AttentionScaling.NONE,
        **common_params,
    )
    unbiased_perceptron_len_scale = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.UNBIASED_PERCEPTRON,
        scaling=AttentionScaling.SEQ_LEN,
        **common_params,
    )
    perceptron = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.PERCEPTRON,
        scaling=AttentionScaling.NONE,
        **common_params,
    )
    perceptron_len_scale = lambda: TextClassifier(
        num_heads=1,
        attention_type=AttentionType.PERCEPTRON,
        scaling=AttentionScaling.SEQ_LEN,
        **common_params,
    )
    collate_fn = (
        lambda samples: collate_text_classification_samples(
            vocabulary, samples, num_classes
        )
    )
    batch_size = 2
    num_grad_acc_batches = 16
    random_seed_base = 424242
    train_batch_fn_basic = lambda epoch: generate_batches(
        epoch=epoch,
        samples=train,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
    )
    train_batch_fn_drop_in = lambda epoch: generate_batches_drop_in(
        model=model, # Late binding!
        epoch=epoch,
        train_batch_fn=train_batch_fn_basic,
        drop_in_schedule=(
            (0,     0, 0.20),
            (0,  9000, 0.30),
            (0, 16000, 0.40),
            (0, 23000, 0.50),
            (0, 30000, 0.60),
            (0, 37000, 0.70),
            (0,  5000, 0.80),
            (1, 12000, 0.90),
            (1, 19000, 1.00),
        ),
        log_path=log_path,
    )
    val_batch_fn = lambda epoch: generate_batches(
        epoch=epoch,
        samples=val,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )
    test_batch_fn = lambda epoch: generate_batches(
        epoch=epoch,
        samples=test,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )

    models = (
        ("Softmax", "list-ops-softmax", train_batch_fn_basic, softmax),
        ("ReLU", "list-ops-relu", train_batch_fn_basic, relu),
        ("ReLU/seql", "list-ops-relu-seql", train_batch_fn_basic, relu_len_scale),
        ("SReLU", "list-ops-simplified-relu", train_batch_fn_basic, simplified_relu),
        ("SReLU/noscl", "list-ops-simplified-relu-no-scale", train_batch_fn_basic, simplified_relu_no_scale),
        ("SReLU/seql", "list-ops-simplified-relu-seql", train_batch_fn_basic, simplified_relu_len_scale),
        ("UbPerc", "list-ops-unbiased-perceptron", train_batch_fn_basic, unbiased_perceptron),
        ("UbPerc/seql", "list-ops-unbiased-perceptron-seql", train_batch_fn_basic, unbiased_perceptron_len_scale),
        ("Perc", "list-ops-perceptron", train_batch_fn_basic, perceptron),
        ("Perc/seql", "list-ops-perceptron-seql", train_batch_fn_basic, perceptron_len_scale),
        ("Softmax+DI", "list-ops-softmax-drop-in", train_batch_fn_drop_in, softmax),
        ("ReLU+DI", "list-ops-relu-drop-in", train_batch_fn_drop_in, relu),
        ("ReLU/seql+DI", "list-ops-relu-seql-drop-in", train_batch_fn_drop_in, relu_len_scale),
        ("SReLU+DI", "list-ops-simplified-relu-drop-in", train_batch_fn_drop_in, simplified_relu),
        ("SReLU/noscl+DI", "list-ops-simplified-relu-no-scale-drop-in", train_batch_fn_drop_in, simplified_relu_no_scale),
        ("SReLU/seql+DI", "list-ops-simplified-relu-seql-drop-in", train_batch_fn_drop_in, simplified_relu_len_scale),
        ("UbPerc+DI", "list-ops-unbiased-perceptron-drop-in", train_batch_fn_drop_in, unbiased_perceptron),
        ("UbPerc/seql+DI", "list-ops-unbiased-perceptron-seql-drop-in", train_batch_fn_drop_in, unbiased_perceptron_len_scale),
        ("Perc+DI", "list-ops-perceptron-drop-in", train_batch_fn_drop_in, perceptron),
        ("Perc/seql+DI", "list-ops-perceptron-seql-drop-in", train_batch_fn_drop_in, perceptron_len_scale),
    )

    metrics = {}

    for model_name, snapshot_dir_base, train_batch_fn, make_model in models:
        for i in range(3):
            log_msg(f"ListOps / {model_name} ({i + 1})", log_path=log_path)
            set_random_seed(12345 + 42 * (i + 1), log_path=log_path)
            model = make_model()
            snapshot_dir = MODELS_PATH / f"{snapshot_dir_base}-{i}"
            _, epoch_times = run_training(
                model,
                loss_fn=cross_entropy_loss,
                train_batch_fn=train_batch_fn,
                test_batch_fn=val_batch_fn,
                num_epochs=num_epochs,
                num_grad_acc_batches=num_grad_acc_batches,
                num_status_batches=1000,
                snapshot_dir=snapshot_dir,
                num_snapshot_batches=10000,
                lr=5e-4,
                log_path=log_path,
            )
            eval_classification_experiment(
                model_name=model_name,
                test=test,
                snapshot_dir=snapshot_dir,
                model=model,
                test_batch_fn=test_batch_fn,
                collate_fn=collate_fn,
                size_key_filters={".indices": False, "embedding.": False},
                epoch_times=epoch_times,
                metrics=metrics,
                log_path=log_path,
            )

    print_classification_experiments_summary(metrics)

    seq_len_mean, seq_len_std = compute_seq_lens(test, collate_fn)

    return metrics, (num_layers, dims, num_heads, seq_len_mean, seq_len_std)

#### WikiText2 experiment

In [36]:
def run_wikitext2():
    log_path = RESULTS_PATH / "wikitext2.txt"
    num_epochs = 20
    seq_len = 256
    num_layers = 12
    dims = 192
    num_heads = 6
    init_experiment(log_path=log_path)
    train, val, test = load_wikitext2()
    vocabulary = Vocabulary(texts=train)
    common_params = dict(
        vocabulary_size=len(vocabulary),
        num_layers=num_layers,
        dims=dims,
        dims_ffn=4 * dims,
    )
    softmax = lambda: CausalLanguageModel(
        num_heads=num_heads,
        attention_type=AttentionType.SOFTMAX,
        **common_params,
    )
    relu = lambda: CausalLanguageModel(
        num_heads=num_heads,
        attention_type=AttentionType.RELU,
        **common_params,
    )
    relu_len_scale = lambda: CausalLanguageModel(
        num_heads=num_heads,
        attention_type=AttentionType.RELU,
        scaling=AttentionScaling.SEQ_LEN,
        **common_params,
    )
    simplified_relu = lambda: CausalLanguageModel(
        num_heads=1,
        attention_type=AttentionType.SIMPLIFIED_RELU,
        **common_params,
    )
    simplified_relu_no_scale = lambda: CausalLanguageModel(
        num_heads=1,
        attention_type=AttentionType.SIMPLIFIED_RELU,
        scaling=AttentionScaling.NONE,
        **common_params,
    )
    simplified_relu_len_scale = lambda: CausalLanguageModel(
        num_heads=1,
        attention_type=AttentionType.SIMPLIFIED_RELU,
        scaling=AttentionScaling.SEQ_LEN,
        **common_params,
    )

    batch_size = 8
    num_grad_acc_batches = 32
    random_seed_base = 424242
    train_batch_fn_basic = lambda epoch: generate_causal_lm_batches(
        epoch=epoch,
        texts=train,
        seq_len=seq_len,
        vocabulary=vocabulary,
        batch_size=batch_size,
        shuffle=True,
    )
    train_batch_fn_drop_in = lambda epoch: generate_batches_drop_in(
        model=model, # Late binding!
        epoch=epoch,
        train_batch_fn=train_batch_fn_basic,
        drop_in_schedule=(
            ( 0, 0, 0.2),
            ( 5, 0, 0.3),
            ( 6, 0, 0.4),
            ( 7, 0, 0.5),
            ( 8, 0, 0.6),
            ( 9, 0, 0.7),
            (10, 0, 0.8),
            (11, 0, 0.9),
            (12, 0, 1.0),
        ),
        log_path=log_path,
    )
    val_batch_fn = lambda epoch: generate_causal_lm_batches(
        epoch=epoch,
        texts=val,
        seq_len=seq_len,
        vocabulary=vocabulary,
        batch_size=batch_size,
        shuffle=False,
    )
    test_batch_fn = lambda epoch: generate_causal_lm_batches(
        epoch=epoch,
        texts=test,
        seq_len=seq_len,
        vocabulary=vocabulary,
        batch_size=batch_size,
        shuffle=False,
    )

    models = (
        ("Softmax", "wikitext2-softmax", train_batch_fn_basic, softmax),
        ("ReLU", "wikitext2-relu", train_batch_fn_basic, relu),
        ("ReLU/seql", "wikitext2-relu-seql", train_batch_fn_basic, relu_len_scale),
        ("SReLU", "wikitext2-simplified-relu", train_batch_fn_basic, simplified_relu),
        ("SReLU/noscl", "wikitext2-simplified-relu-no-scale", train_batch_fn_basic, simplified_relu_no_scale),
        ("SReLU/seql", "wikitext2-simplified-relu-seql", train_batch_fn_basic, simplified_relu_len_scale),
        ("Softmax+DI", "wikitext2-softmax-drop-in", train_batch_fn_drop_in, softmax),
        ("ReLU+DI", "wikitext2-relu-drop-in", train_batch_fn_drop_in, relu),
        ("ReLU/seql+DI", "wikitext2-relu-seql-drop-in", train_batch_fn_drop_in, relu_len_scale),
        ("SReLU+DI", "wikitext2-simplified-relu-drop-in", train_batch_fn_drop_in, simplified_relu),
        ("SReLU/noscl+DI", "wikitext2-simplified-relu-no-scale-drop-in", train_batch_fn_drop_in, simplified_relu_no_scale),
        ("SReLU/seql+DI", "wikitext2-simplified-relu-seql-drop-in", train_batch_fn_drop_in, simplified_relu_len_scale),
    )

    metrics = {}

    for model_name, snapshot_dir_base, train_batch_fn, make_model in models:
        for i in range(3):
            log_msg(f"WikiText2 / {model_name} ({i + 1})", log_path=log_path)
            set_random_seed(12345 + 42 * (i + 1), log_path=log_path)
            model = make_model()
            snapshot_dir = MODELS_PATH / f"{snapshot_dir_base}-{i}"
            _, epoch_times = run_training(
                model,
                loss_fn=valid_cross_entropy_loss,
                train_batch_fn=train_batch_fn,
                test_batch_fn=val_batch_fn,
                num_epochs=num_epochs,
                num_grad_acc_batches=num_grad_acc_batches,
                num_status_batches=20,
                snapshot_dir=snapshot_dir,
                num_snapshot_batches=0,
                lr=5e-4,
                log_path=log_path,
            )
            eval_causal_language_model_experiment(
                model_name=model_name,
                test=test,
                snapshot_dir=snapshot_dir,
                model=model,
                vocabulary=vocabulary,
                test_batch_fn=test_batch_fn,
                epoch_times=epoch_times,
                metrics=metrics,
                size_key_filters={
                    ".indices": False,
                    "embedding.": False,
                    "output.": False,
                },
                log_path=log_path,
            )

    print_causal_language_models_experiments_summary(metrics)

    seq_len_mean, seq_len_std = float(seq_len), 0.0

    return metrics, (num_layers, dims, num_heads, seq_len_mean, seq_len_std)

### Results summary

In [ ]:
def run_and_print_experiment_results():
    def run_or_replay(experiment_name: str, experiment_func: typing.Callable):
        replay = Replay(RESULTS_PATH / f"{experiment_name}-results.json.gz")
        results = replay(experiment_name, experiment_func)
        replay.save()

        return results

    experiments = {}

    is_f1_better = lambda candidate, champion: champion is None or candidate > champion
    is_perplexity_better = lambda candidate, champion: champion is None or candidate < champion

    experiments["AG's News"] = (
        run_or_replay("agnews", run_agnews), "f1", is_f1_better,
    )
    experiments["IMDb"] = (
        run_or_replay("imdb", run_imdb), "f1", is_f1_better,
    )
    experiments["ListOps"] = (
        run_or_replay("listops", run_listops), "f1", is_f1_better,
    )
    experiments["CIFAR-10"] = (
        run_or_replay("cifar10", run_cifar10), "f1", is_f1_better,
    )
    experiments["CIFAR-100"] = (
        run_or_replay("cifar100", run_cifar100), "f1", is_f1_better,
    )
    experiments["IMDb-gen"] = (
        run_or_replay("imdbgen", run_imdbgen), "perplexity", is_perplexity_better,
    )
    experiments["WikiText2"] = (
        run_or_replay("wikitext2", run_wikitext2), "perplexity", is_perplexity_better,
    )

    model_names = []
    seen_model_names = set()

    for experiment, _, _ in experiments.values():
        for model_name in experiment[0].keys():
            if model_name in seen_model_names:
                continue

            seen_model_names.add(model_name)
            model_names.append(model_name)

    experiment_names = list(experiments.keys())
    table = [
        ["-" for _ in experiment_names]
        for model_name in model_names
    ]
    
    for j, experiment_name in enumerate(experiment_names):
        (experiment_results, _), score, is_better = experiments[experiment_name]
        best_scores = [(None, 0, 0), (None, 0, 0)]

        for i, model_name in enumerate(model_names):
            model_results = experiment_results.get(model_name, None)

            if model_results is None:
                continue

            size = model_results["size"][0] / 1_000_000
            scores = mlx.core.array(model_results[score])
            mean = float(scores.mean().item())
            std = float(scores.std().item())

            if is_better(mean, best_scores[-1][0]):
                best_scores.append((mean, i, j))
                best_scores = best_scores[-2:]

            elif is_better(mean, best_scores[-2][0]):
                best_scores[-2] = (mean, i, j)

            table[i][j] = f"{mean:.3f}&nbsp;<small>±&nbsp;{std:.3f}</small>&nbsp;({size:.1f}M)"

        if len(best_scores) > 1:
            _, i, j = best_scores[-1]
            table[i][j] = "**" + table[i][j] + "**"

            _, i, j = best_scores[-2]
            table[i][j] = "<u>" + table[i][j] + "</u>"

    ds_stats = []

    for j, experiment_name in enumerate(experiment_names):
        (_, ds_stat), _, _ = experiments[experiment_name]
        num_layers, dims, num_heads, seq_len_mean, seq_len_std = ds_stat
        ds_stats.append(
            f"*{num_layers}×{dims}/{num_heads}, "
            f"{seq_len_mean:.1f}&nbsp;<small>±&nbsp;{seq_len_std:.1f}</small>*"
        )

    print("| Model | " + (" | ".join(experiment_names)) + " |")
    print("| :--- | " + (" | ".join(":---:" for _ in experiment_names)) + " |")

    for i, row in enumerate(table):
        print(f"| {model_names[i]} | " + (" | ".join(row)) + " |")

    print("| *Layers×Dims/Heads, SeqLen* | " + (" | ".join(ds_stats)) + " |")


run_and_print_experiment_results()